In [ ]:
import pandas as pd, glob, os

folder = "Data/VED_DynamicData_Part1"
files = sorted(glob.glob(os.path.join(folder, "VED_*_week.csv")))[:3]
print("Files:", [os.path.basename(f) for f in files])

dyn = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

print("Rows:", len(dyn), "| vehicles:", dyn["VehId"].nunique())
print("\nMissing share per column:")
print(dyn.isna().mean().round(2).sort_values())

fuel = dyn.dropna(subset=["Fuel Rate[L/hr]"])
print("\nRows with fuel rate:", len(fuel), "| vehicles:", fuel["VehId"].nunique())

per_veh = dyn.groupby("VehId")["Fuel Rate[L/hr]"].apply(lambda s: s.notna().mean())
print("Vehicles with any fuel rate:", (per_veh > 0).sum(), "of", len(per_veh))

In [2]:
import sys
sys.stdout = open(r"D:\Ml project\dataset_report.txt", "w", encoding="utf-8")
import os, glob
import pandas as pd

ROOT = r"D:\Ml project\Data"

def count_rows(path):
    """Fast line count (minus header) without loading the file."""
    n = 0
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1024 * 1024 * 8), b""):
            n += chunk.count(b"\n")
    return max(n - 1, 0)

def describe(name, rows, head_df):
    nulls = (head_df.isna().mean() * 100).round(1)
    print("=" * 70)
    print(f"DATASET : {name}")
    print(f"ROWS    : {rows:,}")
    print(f"COLUMNS : {head_df.shape[1]}")
    for c in head_df.columns:
        print(f"   - {c} | {head_df[c].dtype} | {nulls[c]}% null (first 5000 rows)")

# 1) Single CSV files in the Data folder
for f in sorted(glob.glob(os.path.join(ROOT, "*.csv"))):
    try:
        head = pd.read_csv(f, nrows=5000, encoding_errors="ignore")
        describe(os.path.basename(f), count_rows(f), head)
    except Exception as e:
        print("=" * 70); print("FAILED:", os.path.basename(f), "|", e)

# 2) Excel files
for f in sorted(glob.glob(os.path.join(ROOT, "*.xlsx"))):
    try:
        df = pd.read_excel(f)
        describe(os.path.basename(f), len(df), df)
    except Exception as e:
        print("=" * 70); print("FAILED:", os.path.basename(f), "|", e)

# 3) Folders of many files (VED parts, OBD-II-Dataset, etc.)
for folder in sorted(p for p in glob.glob(os.path.join(ROOT, "*")) if os.path.isdir(p)):
    files = sorted(glob.glob(os.path.join(folder, "**", "*.csv"), recursive=True))
    if not files:
        print("=" * 70); print("FOLDER (no csv found):", os.path.basename(folder)); continue
    try:
        total = sum(count_rows(f) for f in files)
        head = pd.read_csv(files[0], nrows=5000, encoding_errors="ignore")
        describe(f"{os.path.basename(folder)}  [{len(files)} csv files]", total, head)
    except Exception as e:
        print("=" * 70); print("FAILED folder:", os.path.basename(folder), "|", e)

# 4) Weather file you pulled
w = os.path.join(ROOT, "weather_annarbor_utc.csv")
if os.path.exists(w):
    wd = pd.read_csv(w)
    describe("weather_annarbor_utc.csv (Open-Meteo API)", len(wd), wd)

In [3]:
import os, glob, json, random
import numpy as np, pandas as pd

ROOT = r"D:\Ml project\Data"
OUT  = os.path.join(ROOT, "dataset_profile.json")
random.seed(42)

def count_rows(path):
    n = 0
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(8 * 1024 * 1024), b""):
            n += chunk.count(b"\n")
    return max(n - 1, 0)

def load_csv_sample(path, target=200_000):
    n = count_rows(path)
    if n > 300_000:                       # big file: random ~200k-row sample
        keep = target / n
        df = pd.read_csv(path, skiprows=lambda i: i > 0 and random.random() > keep,
                         encoding_errors="ignore", low_memory=False)
    else:                                 # small file: read everything
        df = pd.read_csv(path, encoding_errors="ignore", low_memory=False)
    return n, df

def stats(n):
    n = n.dropna()
    if n.empty: return None
    return {"min": round(float(n.min()), 3), "max": round(float(n.max()), 3),
            "mean": round(float(n.mean()), 3), "median": round(float(n.median()), 3)}

def profile(name, total_rows, df, note=""):
    out = {"dataset": name, "total_rows": int(total_rows), "sample_rows": int(len(df)),
           "columns_count": int(df.shape[1]), "note": note,
           "duplicate_rows_in_sample": int(df.duplicated().sum()),
           "sample_head": [{str(k): str(v)[:40] for k, v in r.items()}
                           for r in df.head(3).to_dict(orient="records")],
           "columns": []}
    for c in df.columns:
        s = df[c]
        info = {"name": str(c), "dtype": str(s.dtype),
                "pct_null": round(float(s.isna().mean() * 100), 1),
                "n_unique": int(s.nunique(dropna=True))}
        if pd.api.types.is_numeric_dtype(s) and not pd.api.types.is_bool_dtype(s):
            info["stats"] = stats(s)
        else:
            parsed = pd.to_numeric(s, errors="coerce")
            if s.notna().sum() > 0 and parsed.notna().sum() / s.notna().sum() >= 0.8:
                info["numeric_stored_as_text"] = True
                info["stats"] = stats(parsed)
            else:
                info["top_values"] = {str(k)[:40]: int(v) for k, v in s.value_counts().head(3).items()}
        out["columns"].append(info)
    return out

report = []

# 1) single CSV files
for f in sorted(glob.glob(os.path.join(ROOT, "*.csv"))):
    name = os.path.basename(f)
    if name == "weather_annarbor.csv":    # local-time duplicate, skip
        continue
    try:
        n, df = load_csv_sample(f)
        report.append(profile(name, n, df, "random sample" if n > 300_000 else "full file"))
        print("done:", name)
    except Exception as e:
        print("FAILED:", name, e)

# 2) Excel
for f in sorted(glob.glob(os.path.join(ROOT, "*.xlsx"))):
    try:
        df = pd.read_excel(f)
        report.append(profile(os.path.basename(f), len(df), df, "full file"))
        print("done:", os.path.basename(f))
    except Exception as e:
        print("FAILED:", f, e)

# 3) VED folders: sample first, middle and last weekly file (100k rows each)
for folder in sorted(glob.glob(os.path.join(ROOT, "VED_DynamicData_Part*"))):
    files = sorted(glob.glob(os.path.join(folder, "*.csv")))
    if not files: continue
    try:
        total = sum(count_rows(f) for f in files)
        pick = [files[0], files[len(files)//2], files[-1]]
        df = pd.concat([pd.read_csv(f, nrows=100_000) for f in pick], ignore_index=True)
        report.append(profile(os.path.basename(folder), total, df,
                              f"{len(files)} files; sample from first, middle, last file"))
        print("done:", os.path.basename(folder))
    except Exception as e:
        print("FAILED:", folder, e)

with open(OUT, "w", encoding="utf-8") as fh:
    json.dump(report, fh, indent=1, default=str)
print("saved:", OUT, "| size:", os.path.getsize(OUT), "bytes")

In [1]:
import pandas as pd
from pathlib import Path

VA = pd.read_pickle(Path("Data") / "ved_features.pkl")
print(VA.shape)
print(VA.dtypes.to_string())
print(VA.head(3).T.to_string())
print(VA.groupby("car").size().describe())

(10203018, 16)
VehId            int64
speed          float32
maf            float32
rpm            float32
dt             float32
speed_sq       float32
rpm_per_kmh    float32
is_idle        float32
is_highway     float32
accel          float32
accel_abs      float32
rpm_change     float32
speed_roll5    float32
rpm_roll5      float32
speed_std5     float32
disp_L         float64
                       0            1            2
VehId           8.000000     8.000000     8.000000
speed          40.000000    45.000000    47.000000
maf            22.129999    22.129999     6.150000
rpm          2285.000000  2285.000000  2744.000000
dt              0.200000     0.900000     1.000000
speed_sq     1600.000000  2025.000000  2209.000000
rpm_per_kmh    57.125000    50.777779    58.382980
is_idle         0.000000     0.000000     0.000000
is_highway      0.000000     0.000000     0.000000
accel                NaN     5.555555     2.000000
accel_abs            NaN     5.555555     2.000000
rpm_c

KeyError: 'car'

In [2]:
from pathlib import Path
import pandas as pd

for f in sorted(Path("Data").glob("*.csv")):
    try:
        n = sum(1 for _ in open(f, "rb")) - 1
        cols = pd.read_csv(f, nrows=0).columns.tolist()
        print(f"{f.name:55s} rows≈{n:>10,}  cols={len(cols)}")
        print("   ", cols[:12])
    except Exception as e:
        print(f.name, "ERR", e)

allcars.csv                                             rows≈ 7,214,695  cols=18
    ['Unnamed: 0', 'timeStamp', 'tripID', 'accData', 'gps_speed', 'battery', 'cTemp', 'dtc', 'eLoad', 'iat', 'imap', 'kpl']
CAR_Prediction_System_DATASET.csv                       rows≈    19,473  cols=29
    ['S.No', 'NAME', 'Price', 'Model', 'Mileage', 'Transmission', 'Engine Type', 'Registration', 'Color', 'Assembled', 'Engine Capacity', 'Body type']
database.csv                                            rows≈    38,113  cols=81
    ['Vehicle ID', 'Year', 'Make', 'Model', 'Class', 'Drive', 'Transmission', 'Transmission Descriptor', 'Engine Index', 'Engine Descriptor', 'Engine Cylinders', 'Engine Displacement']
engine_data.csv                                         rows≈    19,535  cols=7
    ['Engine rpm', 'Lub oil pressure', 'Fuel pressure', 'Coolant pressure', 'lub oil temp', 'Coolant temp', 'Engine Condition']
exp1_14drivers_14cars_dailyRoutes.csv                   rows≈    60,439  cols=33
    ['TI

In [3]:
import pandas as pd
a = pd.read_csv("Data/allcars.csv", nrows=500000)
print(a.columns.tolist())
print(a[["gps_speed","kpl","eLoad"]].describe())
print(a["tripID"].nunique(), a.isna().mean().round(2).to_dict())

e = pd.read_csv("Data/exp1_14drivers_14cars_dailyRoutes.csv", nrows=0)
print(e.columns.tolist())

['Unnamed: 0', 'timeStamp', 'tripID', 'accData', 'gps_speed', 'battery', 'cTemp', 'dtc', 'eLoad', 'iat', 'imap', 'kpl', 'maf', 'rpm', 'speed', 'tAdv', 'tPos', 'deviceID']
           gps_speed            kpl          eLoad
count  380212.000000  380212.000000  380212.000000
mean        8.261171       6.985664      36.487072
std        13.946388       9.109200      22.048786
min         0.000000       0.000000       0.000000
25%         0.000000       0.000000      29.019608
50%         0.000000       0.000000      29.019608
75%        13.300000      18.005960      47.450980
max        82.100000    1095.112500      99.607843
129 {'Unnamed: 0': 0.0, 'timeStamp': 0.0, 'tripID': 0.0, 'accData': 0.0, 'gps_speed': 0.24, 'battery': 0.24, 'cTemp': 0.79, 'dtc': 0.79, 'eLoad': 0.24, 'iat': 0.79, 'imap': 0.79, 'kpl': 0.24, 'maf': 0.24, 'rpm': 0.24, 'speed': 0.0, 'tAdv': 0.79, 'tPos': 0.24, 'deviceID': 0.0}
['TIMESTAMP', 'MARK', 'MODEL', 'CAR_YEAR', 'ENGINE_POWER', 'AUTOMATIC', 'VEHICLE_ID', 'BAROME

In [4]:
del VA
import gc; gc.collect()

798

In [1]:
import pandas as pd, numpy as np
from pathlib import Path
root = Path("Data")

M = pd.read_csv(root / "measurements.csv", decimal=",")
for c in ["distance", "consume", "speed", "temp_inside", "temp_outside", "refill liters"]:
    M[c] = pd.to_numeric(M[c], errors="coerce")
print(M.shape)
print(M.dtypes)
print(M.head(10).to_string())
print(M.tail(5).to_string())

(388, 12)
distance         float64
consume          float64
speed              int64
temp_inside      float64
temp_outside       int64
specials             str
gas_type             str
AC                 int64
rain               int64
sun                int64
refill liters    float64
refill gas           str
dtype: object
   distance  consume  speed  temp_inside  temp_outside specials gas_type  AC  rain  sun  refill liters refill gas
0      28.0      5.0     26         21.5            12      NaN      E10   0     0    0           45.0        E10
1      12.0      4.2     30         21.5            13      NaN      E10   0     0    0            NaN        NaN
2      11.2      5.5     38         21.5            15      NaN      E10   0     0    0            NaN        NaN
3      12.9      3.9     36         21.5            14      NaN      E10   0     0    0            NaN        NaN
4      18.5      4.5     46         21.5            15      NaN      E10   0     0    0            NaN    

In [2]:
import pandas as pd
from pathlib import Path
root = Path("Data")
files = ["exp1_14drivers_14cars_dailyRoutes.csv", "exp2_19drivers_1car_1route.csv",
         "exp3_4drivers_1car_1route.csv", "allcars.csv", "engine_data.csv", "measurements.csv"]
for f in files:
    p = root / f
    n = sum(1 for _ in open(p, "rb")) - 1
    cols = list(pd.read_csv(p, nrows=1).columns)
    print(f"{f}: ~{n:,} rows (line count), {len(cols)} columns")
    print("   ", cols[:12])

exp1_14drivers_14cars_dailyRoutes.csv: ~60,439 rows (line count), 33 columns
    ['TIMESTAMP', 'MARK', 'MODEL', 'CAR_YEAR', 'ENGINE_POWER', 'AUTOMATIC', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE(KPA)', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP']
exp2_19drivers_1car_1route.csv: ~8,261 rows (line count), 28 columns
    ['TIME', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE']
exp3_4drivers_1car_1route.csv: ~1,743 rows (line count), 30 columns
    ['ord', 'TIME', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM']
allcars.csv: ~7,214,695 rows (line count), 18 columns
    ['Unnamed: 0', 'timeStamp', 'tripID', 'accData', 'gps_speed', 'battery', 'cTemp', 'dtc', 'eLoad', 'iat', 'imap', 'kpl']
engine_data.csv: ~19,535 rows (line c

In [3]:
import pandas as pd, numpy as np
from pathlib import Path
root = Path("Data")

e1 = pd.read_csv(root / "exp1_14drivers_14cars_dailyRoutes.csv", low_memory=False)
print(e1.shape)
print(list(e1.columns))

spec = [c for c in ["MARK", "MODEL", "CAR_YEAR", "ENGINE_POWER", "AUTOMATIC"] if c in e1.columns]
g = e1.groupby("VEHICLE_ID")
out = g[spec].first()
out["rows"] = g.size()
maf = [c for c in e1.columns if c.upper() == "MAF"]
if maf:
    out["MAF_filled"] = g[maf[0]].apply(lambda z: pd.to_numeric(z, errors="coerce").notna().mean()).round(2)
print(out.to_string())

(60439, 33)
['TIMESTAMP', 'MARK', 'MODEL', 'CAR_YEAR', 'ENGINE_POWER', 'AUTOMATIC', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE(KPA)', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE', 'MAF', 'LONG TERM FUEL TRIM BANK 2', 'FUEL_TYPE', 'AIR_INTAKE_TEMP', 'FUEL_PRESSURE', 'SPEED', 'SHORT TERM FUEL TRIM BANK 2', 'SHORT TERM FUEL TRIM BANK 1', 'ENGINE_RUNTIME', 'THROTTLE_POS', 'DTC_NUMBER', 'TROUBLE_CODES', 'TIMING_ADVANCE', 'EQUIV_RATIO', 'MIN', 'HOURS', 'DAYS_OF_WEEK', 'MONTHS', 'YEAR']
                  MARK       MODEL  CAR_YEAR ENGINE_POWER AUTOMATIC   rows  MAF_filled
VEHICLE_ID                                                                            
car1         chevrolet       agile    2011.0          1,4         n  13617        0.00
car10             fiat       siena    2010.0            1         n      7        0.00
car11           toyota     corolla    2009.0          1,8         s   7485        0.01
car12           niss

In [4]:
raw = pd.read_csv(root / "exp1_14drivers_14cars_dailyRoutes.csv", low_memory=False, dtype=str)
g = raw.groupby("VEHICLE_ID")
chk = pd.DataFrame({
    "rows": g.size(),
    "MAF_filled": g["MAF"].apply(lambda z: z.notna().mean()).round(2),
    "RPM_filled": g["ENGINE_RPM"].apply(lambda z: z.notna().mean()).round(2),
    "SPEED_filled": g["SPEED"].apply(lambda z: z.notna().mean()).round(2),
})
print(chk.to_string())
print(raw["MAF"].dropna().head(5).tolist())
print(raw.groupby("VEHICLE_ID")["FUEL_TYPE"].agg(lambda z: z.dropna().unique()[:3].tolist()).to_string())

             rows  MAF_filled  RPM_filled  SPEED_filled
VEHICLE_ID                                             
car1        13617        0.20        0.20          1.00
car10           7        0.00        0.71          0.86
car11        7485        0.98        0.98          0.99
car12        1866        0.96        0.96          0.94
car13        2060        0.00        0.87          0.89
car14         180        0.00        1.00          0.99
car2          138        0.00        1.00          0.93
car3          903        0.00        0.98          0.97
car4         2920        0.00        0.98          0.99
car5           55        0.00        1.00          0.95
car6         6184        0.00        0.97          0.99
car7          507        0.00        0.99          0.95
car8         5574        0.00        0.71          1.00
car9         6018        0.00        0.94          0.94
['4,49', '4,51', '4,48', '4,51', '4,49']
VEHICLE_ID
car1      [Biodiesel_Ethanol]
car10                 

In [6]:
from sklearn.ensemble import HistGradientBoostingRegressor

def numc(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")

# ---- test cars from exp1 ----
T = raw[raw["VEHICLE_ID"].isin(["car1", "car11", "car12"])].copy()
for c in ["MAF", "ENGINE_RPM", "SPEED", "ENGINE_LOAD", "ENGINE_POWER", "TIMESTAMP"]:
    T[c] = numc(T[c])
T = T.rename(columns={"MAF": "maf", "ENGINE_RPM": "rpm", "SPEED": "speed",
                      "ENGINE_LOAD": "load", "ENGINE_POWER": "disp_L"})
T = T.sort_values(["VEHICLE_ID", "TIMESTAMP"])
T["dt"] = T.groupby("VEHICLE_ID")["TIMESTAMP"].diff() / 1000
T = T[T["dt"].between(0.1, 30) & T["maf"].between(0.5, 250)
      & T["speed"].notna() & T["rpm"].notna()].copy()

g = T.groupby("VEHICLE_ID")
T["rpm_x_load"] = T["rpm"] * T["load"]
T["speed_sq"] = T["speed"] ** 2
T["rpm_per_kmh"] = (T["rpm"] / T["speed"].clip(lower=1)).clip(upper=500)
T["load_per_rpm"] = T["load"] / T["rpm"].clip(lower=1)
T["is_idle"] = (T["speed"] < 2).astype(int)
T["is_highway"] = (T["speed"] > 80).astype(int)
T["accel"] = (g["speed"].diff() / T["dt"]).clip(-10, 10)
T["accel_abs"] = T["accel"].abs()
T["rpm_change"] = g["rpm"].diff().abs()
T["speed_roll5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=1).mean())
T["rpm_roll5"] = g["rpm"].transform(lambda z: z.rolling(5, min_periods=1).mean())
T["speed_std5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=2).std())
print(T.groupby("VEHICLE_ID").agg(rows=("dt", "size"), disp=("disp_L", "first"),
      load_med=("load", "median"), maf_med=("maf", "median")))

# ---- VED training data with engine displacement ----
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
st["disp_L"] = st["Engine Configuration & Displacement"].str.extract(r"(\d+\.\d+)L")[0].astype(float)
VT = VA.merge(st[["VehId", "disp_L"]], on="VehId", how="left")
VT = VT.dropna(subset=["disp_L", "maf", "rpm", "speed"]).sample(500_000, random_state=0)

base = ["rpm", "speed", "speed_sq", "rpm_per_kmh", "is_idle", "is_highway", "accel",
        "accel_abs", "rpm_change", "speed_roll5", "rpm_roll5", "speed_std5", "disp_L"]
sets = {"no_load": base, "with_load": base + ["load", "rpm_x_load", "load_per_rpm"]}

for label, feats in sets.items():
    m = HistGradientBoostingRegressor(random_state=0).fit(VT[feats], VT["maf"])
    T["pred"] = m.predict(T[feats])
    c = T.assign(air_true=T["maf"] * T["dt"], air_pred=T["pred"] * T["dt"]).groupby("VEHICLE_ID")[["air_true", "air_pred"]].sum()
    c["error_%"] = 100 * (c["air_pred"] - c["air_true"]) / c["air_true"]
    print("\n", label, "(signed error: + means over-predicts airflow)")
    print(c.round(1))

            rows  disp  load_med  maf_med
VEHICLE_ID                               
car1        2675   1.4       NaN     5.19
car11       7222   1.8       NaN    10.76
car12       1608   1.6       NaN     4.30


NameError: name 'VA' is not defined

import glob
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
ice = set(st.loc[st["Vehicle Type"] == "ICE", "VehId"])
st["disp_L"] = st["Engine Configuration & Displacement"].str.extract(r"(\d+\.\d+)L")[0].astype(float)

cols = ["VehId", "Trip", "Timestamp(ms)", "Vehicle Speed[km/h]", "MAF[g/sec]", "Engine RPM[RPM]"]
ren = {"Vehicle Speed[km/h]": "speed", "MAF[g/sec]": "maf", "Engine RPM[RPM]": "rpm"}
files = sorted(glob.glob(str(root / "VED_DynamicData_Part*" / "*.csv")))

def load_week(f):
    w = pd.read_csv(f, usecols=cols).rename(columns=ren)
    w = w[w["VehId"].isin(ice)].sort_values(["VehId", "Trip", "Timestamp(ms)"])
    k = ["VehId", "Trip"]
    w["dt"] = w.groupby(k)["Timestamp(ms)"].diff() / 1000
    w = w[w["dt"].between(0.1, 30) & w["maf"].between(0.5, 250)
          & w["speed"].notna() & w["rpm"].notna()].copy()
    g = w.groupby(k)
    w["speed_sq"] = w["speed"] ** 2
    w["rpm_per_kmh"] = (w["rpm"] / w["speed"].clip(lower=1)).clip(upper=500)
    w["is_idle"] = (w["speed"] < 2).astype(int)
    w["is_highway"] = (w["speed"] > 80).astype(int)
    w["accel"] = (g["speed"].diff() / w["dt"]).clip(-10, 10)
    w["accel_abs"] = w["accel"].abs()
    w["rpm_change"] = g["rpm"].diff().abs()
    w["speed_roll5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=1).mean())
    w["rpm_roll5"] = g["rpm"].transform(lambda z: z.rolling(5, min_periods=1).mean())
    w["speed_std5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=2).std())
    return w.drop(columns=["Timestamp(ms)", "Trip"]).replace([np.inf, -np.inf], np.nan).astype("float32")

VA = pd.concat([load_week(f) for f in files], ignore_index=True)
VA["VehId"] = VA["VehId"].astype(int)
VA = VA.merge(st[["VehId", "disp_L"]], on="VehId", how="left")
VA = VA.dropna(subset=["disp_L"])
VA.to_pickle(root / "ved_features.pkl")
print(VA.shape, "cars:", VA["VehId"].nunique())

In [7]:
import glob
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
ice = set(st.loc[st["Vehicle Type"] == "ICE", "VehId"])
st["disp_L"] = st["Engine Configuration & Displacement"].str.extract(r"(\d+\.\d+)L")[0].astype(float)

cols = ["VehId", "Trip", "Timestamp(ms)", "Vehicle Speed[km/h]", "MAF[g/sec]", "Engine RPM[RPM]"]
ren = {"Vehicle Speed[km/h]": "speed", "MAF[g/sec]": "maf", "Engine RPM[RPM]": "rpm"}
files = sorted(glob.glob(str(root / "VED_DynamicData_Part*" / "*.csv")))

def load_week(f):
    w = pd.read_csv(f, usecols=cols).rename(columns=ren)
    w = w[w["VehId"].isin(ice)].sort_values(["VehId", "Trip", "Timestamp(ms)"])
    k = ["VehId", "Trip"]
    w["dt"] = w.groupby(k)["Timestamp(ms)"].diff() / 1000
    w = w[w["dt"].between(0.1, 30) & w["maf"].between(0.5, 250)
          & w["speed"].notna() & w["rpm"].notna()].copy()
    g = w.groupby(k)
    w["speed_sq"] = w["speed"] ** 2
    w["rpm_per_kmh"] = (w["rpm"] / w["speed"].clip(lower=1)).clip(upper=500)
    w["is_idle"] = (w["speed"] < 2).astype(int)
    w["is_highway"] = (w["speed"] > 80).astype(int)
    w["accel"] = (g["speed"].diff() / w["dt"]).clip(-10, 10)
    w["accel_abs"] = w["accel"].abs()
    w["rpm_change"] = g["rpm"].diff().abs()
    w["speed_roll5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=1).mean())
    w["rpm_roll5"] = g["rpm"].transform(lambda z: z.rolling(5, min_periods=1).mean())
    w["speed_std5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=2).std())
    return w.drop(columns=["Timestamp(ms)", "Trip"]).replace([np.inf, -np.inf], np.nan).astype("float32")

VA = pd.concat([load_week(f) for f in files], ignore_index=True)
VA["VehId"] = VA["VehId"].astype(int)
VA = VA.merge(st[["VehId", "disp_L"]], on="VehId", how="left")
VA = VA.dropna(subset=["disp_L"])
VA.to_pickle(root / "ved_features.pkl")
print(VA.shape, "cars:", VA["VehId"].nunique())

(10203018, 16) cars: 196


In [8]:
from sklearn.ensemble import HistGradientBoostingRegressor

VT = VA.sample(600_000, random_state=0)
feats = ["rpm", "speed", "speed_sq", "rpm_per_kmh", "is_idle", "is_highway", "accel",
         "accel_abs", "rpm_change", "speed_roll5", "rpm_roll5", "speed_std5", "disp_L"]

m = HistGradientBoostingRegressor(random_state=0).fit(VT[feats], VT["maf"])
T["pred"] = m.predict(T[feats])
c = (T.assign(air_true=T["maf"] * T["dt"], air_pred=T["pred"] * T["dt"])
       .groupby("VEHICLE_ID")[["air_true", "air_pred"]].sum())
c["error_%"] = 100 * (c["air_pred"] - c["air_true"]) / c["air_true"]
print("signed error (+ means the model over-predicts airflow)")
print(c.round(1))

row_err = (T["pred"] - T["maf"]).abs().groupby(T["VEHICLE_ID"]).mean().round(2)
print("row-level MAE (g/s):\n", row_err)

signed error (+ means the model over-predicts airflow)
            air_true   air_pred  error_%
VEHICLE_ID                              
car1        168245.7   240841.3     43.1
car11       892843.0  1190194.8     33.3
car12       111041.2   151711.9     36.6
row-level MAE (g/s):
 VEHICLE_ID
car1     4.09
car11    5.68
car12    4.75
dtype: float64


In [9]:
c["ratio"] = c["air_pred"] / c["air_true"]
for car in c.index:
    others = c.drop(car)["ratio"].mean()
    print(car, "error after calibrating on the other two:",
          round(100 * (c.loc[car, "ratio"] / others - 1), 1), "%")

car1 error after calibrating on the other two: 6.1 %
car11 error after calibrating on the other two: -4.7 %
car12 error after calibrating on the other two: -1.2 %


In [10]:
def ve(d): return d["maf"] / (d["disp_L"] * d["rpm"] / 120 * 1.2)
T["ve"] = ve(T)
V2 = VA.sample(600_000, random_state=1).copy(); V2["ve"] = ve(V2)
print("exp1 VE by car:\n", T.groupby("VEHICLE_ID")["ve"].median().round(3))
for lo, hi in [(1.3, 1.5), (1.5, 1.7), (1.7, 1.9)]:
    s = V2[V2["disp_L"].between(lo, hi)]
    print(f"VED {lo}-{hi} L: VE median {s['ve'].median():.3f}, cars {s['VehId'].nunique()}")
print("exp1 median rpm/speed:\n", T.groupby("VEHICLE_ID")[["rpm", "speed"]].median())
print("VED median rpm/speed:", V2[["rpm", "speed"]].median().to_dict())

exp1 VE by car:
 VEHICLE_ID
car1     0.302
car11    0.331
car12    0.230
Name: ve, dtype: float64
VED 1.3-1.5 L: VE median 0.273, cars 11
VED 1.5-1.7 L: VE median 0.273, cars 8
VED 1.7-1.9 L: VE median 0.250, cars 19
exp1 median rpm/speed:
                rpm  speed
VEHICLE_ID               
car1        1480.0   28.0
car11       1877.0   57.5
car12       1287.5   28.5
VED median rpm/speed: {'rpm': 1349.0, 'speed': 41.0}


In [11]:
import pandas as pd, numpy as np, glob
from pathlib import Path
root = Path("Data")

f = glob.glob(str(root / "**" / "MY2022*.csv"), recursive=True)[0]
F = pd.read_csv(f)
print(f, F.shape)
print(list(F.columns))
print(F.head(3).T.to_string())
print("missing:\n", F.isna().sum()[F.isna().sum() > 0])
print("duplicates:", F.duplicated().sum())

Data\MY2022 Fuel Consumption Ratings.csv (946, 15)
['Model Year', 'Make', 'Model', 'Vehicle Class', 'Engine Size(L)', 'Cylinders', 'Transmission', 'Fuel Type', 'Fuel Consumption (City (L/100 km)', 'Fuel Consumption(Hwy (L/100 km))', 'Fuel Consumption(Comb (L/100 km))', 'Fuel Consumption(Comb (mpg))', 'CO2 Emissions(g/km)', 'CO2 Rating', 'Smog Rating']
                                         0           1           2
Model Year                            2022        2022        2022
Make                                 Acura       Acura       Acura
Model                                  ILX  MDX SH-AWD  RDX SH-AWD
Vehicle Class                      Compact  SUV: Small  SUV: Small
Engine Size(L)                         2.4         3.5         2.0
Cylinders                                4           6           4
Transmission                           AM8        AS10        AS10
Fuel Type                                Z           Z           Z
Fuel Consumption (City (L/100 km)      9.9 

In [12]:
import pandas as pd, numpy as np
from pathlib import Path
root = Path("Data")

F = pd.read_csv(root / "database.csv", low_memory=False)
print(F.shape)
for i, c in enumerate(F.columns):
    print(i, c, "| filled:", round(F[c].notna().mean(), 2), "| e.g.", F[c].dropna().iloc[0] if F[c].notna().any() else None)

(38113, 81)
0 Vehicle ID | filled: 1.0 | e.g. 26587
1 Year | filled: 1.0 | e.g. 1984
2 Make | filled: 1.0 | e.g. Alfa Romeo
3 Model | filled: 1.0 | e.g. GT V6 2.5
4 Class | filled: 1.0 | e.g. Minicompact Cars
5 Drive | filled: 0.97 | e.g. 2-Wheel Drive
6 Transmission | filled: 1.0 | e.g. Manual 5-Speed
7 Transmission Descriptor | filled: 0.39 | e.g. Creeper
8 Engine Index | filled: 1.0 | e.g. 9001
9 Engine Descriptor | filled: 0.6 | e.g. (FFS)
10 Engine Cylinders | filled: 1.0 | e.g. 6.0
11 Engine Displacement | filled: 1.0 | e.g. 2.5
12 Turbocharger | filled: 0.14 | e.g. T
13 Supercharger | filled: 0.02 | e.g. S
14 Fuel Type | filled: 1.0 | e.g. Regular
15 Fuel Type 1 | filled: 1.0 | e.g. Regular Gasoline
16 Fuel Type 2 | filled: 0.04 | e.g. Natural Gas
17 City MPG (FT1) | filled: 1.0 | e.g. 17
18 Unrounded City MPG (FT1) | filled: 1.0 | e.g. 0.0
19 City MPG (FT2) | filled: 1.0 | e.g. 0
20 Unrounded City MPG (FT2) | filled: 1.0 | e.g. 0.0
21 City Gasoline Consumption (CD) | filled: 1.

In [13]:
print(F["Fuel Type 1"].value_counts())
print("displacement == 0:", (F["Engine Displacement"] == 0).sum(),
      "| cylinders == 0:", (F["Engine Cylinders"] == 0).sum())

G = F[F["Fuel Type 1"].str.contains("Gasoline|Diesel", na=False)].copy()
city, hwy, comb = "City MPG (FT1)", "Highway MPG (FT1)", "Combined MPG (FT1)"
G = G[(G[city] > 0) & (G[hwy] > 0) & (G[comb] > 0)]
print("gasoline/diesel rows with valid mpg:", len(G))

G["l100"] = 235.215 / G[comb]
print(G["l100"].describe().round(1))

calc = 1 / (0.55 / G[city] + 0.45 / G[hwy])
gap = (calc - G[comb]).abs()
print("combined vs 55/45 harmonic blend: median gap", round(gap.median(), 2),
      "| share within 1 mpg:", round((gap <= 1).mean(), 3), "| max", round(gap.max(), 1))

co2 = G[G["Tailpipe CO2 in Grams/Mile (FT1)"] > 0]
print("CO2 vs L/100km corr:", round(co2["Tailpipe CO2 in Grams/Mile (FT1)"].corr(co2["l100"]), 3),
      "(rows with CO2:", len(co2), ")")

G["decade"] = (G["Year"] // 10) * 10
print(G.groupby("decade")["l100"].agg(["count", "median"]).round(1))

dd = G.drop_duplicates(["Year", "Make", "Model", "Engine Cylinders", "Engine Displacement", "Transmission"])
print("rows after dropping duplicate configurations:", len(dd))
print("outliers (<3 or >40 L/100km):", (~G["l100"].between(3, 40)).sum())


Fuel Type 1
Regular Gasoline     26531
Premium Gasoline     10298
Diesel                1014
Electricity            133
Midgrade Gasoline       77
Natural Gas             60
Name: count, dtype: int64
displacement == 0: 1 | cylinders == 0: 0
gasoline/diesel rows with valid mpg: 37920
count    37920.0
mean        12.5
std          3.1
min          4.1
25%         10.2
50%         12.4
75%         13.8
max         33.6
Name: l100, dtype: float64
combined vs 55/45 harmonic blend: median gap 0.24 | share within 1 mpg: 1.0 | max 1.0
CO2 vs L/100km corr: 0.995 (rows with CO2: 37920 )
        count  median
decade               
1980     8405    12.4
1990     9549    12.4
2000    10602    12.4
2010     9364    11.2
rows after dropping duplicate configurations: 33389
outliers (<3 or >40 L/100km): 0


VA = pd.read_pickle(root / "ved_features.pkl")
c = VA.groupby("VehId").apply(lambda d: pd.Series({
    "disp": d["disp_L"].iloc[0],
    "L": (d["maf"] / 14.7 / 750 * d["dt"]).sum(),
    "km": (d["speed"] * d["dt"] / 3600).sum()}))
c = c[c["km"] > 20]
c["l100_ved"] = 100 * c["L"] / c["km"]
c["bin"] = pd.cut(c["disp"], [0, 1.5, 2.0, 2.6, 3.6, 4.6, 9])

E = G[G["Fuel Type 1"].str.contains("Gasoline") & (G["Year"] >= 2005)].copy()
E["bin"] = pd.cut(E["Engine Displacement"], [0, 1.5, 2.0, 2.6, 3.6, 4.6, 9])

out = pd.DataFrame({"VED_cars": c.groupby("bin", observed=True).size(),
                    "VED_median": c.groupby("bin", observed=True)["l100_ved"].median(),
                    "EPA_median": E.groupby("bin", observed=True)["l100"].median()}).round(1)
out["VED/EPA"] = (out["VED_median"] / out["EPA_median"]).round(2)
print(out)

In [14]:
VA = pd.read_pickle(root / "ved_features.pkl")
c = VA.groupby("VehId").apply(lambda d: pd.Series({
    "disp": d["disp_L"].iloc[0],
    "L": (d["maf"] / 14.7 / 750 * d["dt"]).sum(),
    "km": (d["speed"] * d["dt"] / 3600).sum()}))
c = c[c["km"] > 20]
c["l100_ved"] = 100 * c["L"] / c["km"]
c["bin"] = pd.cut(c["disp"], [0, 1.5, 2.0, 2.6, 3.6, 4.6, 9])

E = G[G["Fuel Type 1"].str.contains("Gasoline") & (G["Year"] >= 2005)].copy()
E["bin"] = pd.cut(E["Engine Displacement"], [0, 1.5, 2.0, 2.6, 3.6, 4.6, 9])

out = pd.DataFrame({"VED_cars": c.groupby("bin", observed=True).size(),
                    "VED_median": c.groupby("bin", observed=True)["l100_ved"].median(),
                    "EPA_median": E.groupby("bin", observed=True)["l100"].median()}).round(1)
out["VED/EPA"] = (out["VED_median"] / out["EPA_median"]).round(2)
print(out)

            VED_cars  VED_median  EPA_median  VED/EPA
bin                                                  
(0.0, 1.5]         8         7.5         7.4     1.01
(1.5, 2.0]        40         8.2         9.0     0.91
(2.0, 2.6]        67         9.3        10.2     0.91
(2.6, 3.6]        41        12.0        11.8     1.02
(3.6, 4.6]         6        13.3        13.1     1.02
(4.6, 9.0]         9        16.8        15.7     1.07


In [15]:
import pandas as pd, numpy as np, glob
from pathlib import Path
root = Path("Data")

f = glob.glob(str(root / "**" / "CAR_Prediction*.csv"), recursive=True)[0]
P = pd.read_csv(f, low_memory=False)
print(P.shape)
for i, c in enumerate(P.columns):
    print(i, c, "| filled:", round(P[c].notna().mean(), 2), "| e.g.", P[c].dropna().iloc[0] if P[c].notna().any() else None)

(19473, 29)
0 S.No | filled: 1.0 | e.g. 1
1 NAME | filled: 1.0 | e.g. Honda City 1.3 i-VTEC Prosmatec 2018
2 Price | filled: 1.0 | e.g. PKR 27.95 lacs
3 Model | filled: 1.0 | e.g. ['2018']
4 Mileage | filled: 1.0 | e.g. ['35,218 km']
5 Transmission | filled: 1.0 | e.g. ['Automatic']
6 Engine Type | filled: 1.0 | e.g. ['Petrol']
7 Registration | filled: 1.0 | e.g. ['Islamabad']
8 Color | filled: 1.0 | e.g. ['Black']
9 Assembled | filled: 1.0 | e.g. ['Local']
10 Engine Capacity | filled: 1.0 | e.g. ['1300 cc']
11 Body type | filled: 1.0 | e.g. ['Sedan']
12 Updated on | filled: 1.0 | e.g. ['May 31, 2022']
13 Feature_1 | filled: 1.0 | e.g. [' ABS']
14 Feature_2 | filled: 1.0 | e.g. [' AM/FM Radio']
15 Feature_3 | filled: 1.0 | e.g. [' Air Conditioning']
16 Feature_4 | filled: 1.0 | e.g. [' CD Player']
17 Feature_5 | filled: 1.0 | e.g. [' DVD Player']
18 Feature_6 | filled: 1.0 | e.g. [' Immobilizer Key']
19 Feature_7 | filled: 1.0 | e.g. [' Keyless Entry']
20 Feature_8 | filled: 1.0 | e.g.

In [16]:
def strip(s):
    return s.astype(str).str.replace(r"[\[\]']", "", regex=True).str.strip().replace({"": np.nan, "nan": np.nan})

P["year"] = pd.to_numeric(strip(P["Model"]), errors="coerce")
P["km"] = pd.to_numeric(strip(P["Mileage"]).str.replace(",", "").str.extract(r"(\d+)")[0], errors="coerce")
P["cc"] = pd.to_numeric(strip(P["Engine Capacity"]).str.replace(",", "").str.extract(r"(\d+)")[0], errors="coerce")
P["fuel"] = strip(P["Engine Type"])
P["assembly"] = strip(P["Assembled"])
P["gear"] = strip(P["Transmission"])
P["body"] = strip(P["Body type"])
pr = P["Price"].astype(str)
P["price_pkr"] = (pr.str.extract(r"([\d.]+)\s*lac", expand=False).astype(float) * 1e5
                  ).fillna(pr.str.extract(r"([\d.]+)\s*crore", expand=False).astype(float) * 1e7)
P["make"] = P["NAME"].astype(str).str.split().str[0]

print("rows:", len(P), "| duplicate NAME+year+km+price:", P.duplicated(["NAME", "year", "km", "price_pkr"]).sum())
print("year:", P["year"].min(), "-", P["year"].max(), "| missing:", P["year"].isna().sum())
print("engine cc parsed:", P["cc"].notna().sum(), "| missing:", P["cc"].isna().sum())
print(P["cc"].describe().round(0))
print(P["fuel"].value_counts(dropna=False))
print(P["assembly"].value_counts(dropna=False))
print(P["gear"].value_counts(dropna=False).head(5))
print(P["make"].value_counts().head(12))
print("km parsed:", P["km"].notna().sum(), "| median km:", P["km"].median(),
      "| km < 100:", (P["km"] < 100).sum(), "| km > 400000:", (P["km"] > 400000).sum())
print("price parsed:", P["price_pkr"].notna().sum(), "| 'Call for price':", (pr == "Call for price").sum())

rows: 19473 | duplicate NAME+year+km+price: 2900
year: 1990.0 - 2022.0 | missing: 659
engine cc parsed: 19070 | missing: 403
count    19070.0
mean      1533.0
std        822.0
min         15.0
25%       1000.0
50%       1300.0
75%       1800.0
max      10000.0
Name: cc, dtype: float64
fuel
Petrol      17044
Diesel        986
Hybrid        902
local         444
NaN            49
Electric       48
Name: count, dtype: int64
assembly
Imported    7119
local       6705
Local       5649
Name: count, dtype: int64
gear
Automatic    12174
Manual        6896
local          403
Name: count, dtype: int64
make
Toyota        6907
Suzuki        4459
Honda         3949
Daihatsu       787
KIA            613
Nissan         442
Hyundai        368
Mercedes       335
Mitsubishi     288
MG             252
Changan        220
Audi           176
Name: count, dtype: int64
km parsed: 19070 | median km: 70000.0 | km < 100: 1888 | km > 400000: 90
price parsed: 18303 | 'Call for price': 1146


In [17]:
def strip(s):
    return s.astype(str).str.replace(r"[\[\]']", "", regex=True).str.strip().replace({"": np.nan, "nan": np.nan})

P["year"] = pd.to_numeric(strip(P["Model"]), errors="coerce")
P["km"] = pd.to_numeric(strip(P["Mileage"]).str.replace(",", "").str.extract(r"(\d+)")[0], errors="coerce")
P["cc"] = pd.to_numeric(strip(P["Engine Capacity"]).str.replace(",", "").str.extract(r"(\d+)")[0], errors="coerce")
P["fuel"] = strip(P["Engine Type"])
P["assembly"] = strip(P["Assembled"])
P["gear"] = strip(P["Transmission"])
P["body"] = strip(P["Body type"])
pr = P["Price"].astype(str)
P["price_pkr"] = (pr.str.extract(r"([\d.]+)\s*lac", expand=False).astype(float) * 1e5
                  ).fillna(pr.str.extract(r"([\d.]+)\s*crore", expand=False).astype(float) * 1e7)
P["make"] = P["NAME"].astype(str).str.split().str[0]

print("rows:", len(P), "| duplicate NAME+year+km+price:", P.duplicated(["NAME", "year", "km", "price_pkr"]).sum())
print("year:", P["year"].min(), "-", P["year"].max(), "| missing:", P["year"].isna().sum())
print("engine cc parsed:", P["cc"].notna().sum(), "| missing:", P["cc"].isna().sum())
print(P["cc"].describe().round(0))
print(P["fuel"].value_counts(dropna=False))
print(P["assembly"].value_counts(dropna=False))
print(P["gear"].value_counts(dropna=False).head(5))
print(P["make"].value_counts().head(12))
print("km parsed:", P["km"].notna().sum(), "| median km:", P["km"].median(),
      "| km < 100:", (P["km"] < 100).sum(), "| km > 400000:", (P["km"] > 400000).sum())
print("price parsed:", P["price_pkr"].notna().sum(), "| 'Call for price':", (pr == "Call for price").sum())

rows: 19473 | duplicate NAME+year+km+price: 2900
year: 1990.0 - 2022.0 | missing: 659
engine cc parsed: 19070 | missing: 403
count    19070.0
mean      1533.0
std        822.0
min         15.0
25%       1000.0
50%       1300.0
75%       1800.0
max      10000.0
Name: cc, dtype: float64
fuel
Petrol      17044
Diesel        986
Hybrid        902
local         444
NaN            49
Electric       48
Name: count, dtype: int64
assembly
Imported    7119
local       6705
Local       5649
Name: count, dtype: int64
gear
Automatic    12174
Manual        6896
local          403
Name: count, dtype: int64
make
Toyota        6907
Suzuki        4459
Honda         3949
Daihatsu       787
KIA            613
Nissan         442
Hyundai        368
Mercedes       335
Mitsubishi     288
MG             252
Changan        220
Audi           176
Name: count, dtype: int64
km parsed: 19070 | median km: 70000.0 | km < 100: 1888 | km > 400000: 90
price parsed: 18303 | 'Call for price': 1146


In [18]:
import pandas as pd, glob, os
for f in glob.glob(r"D:\Ml project\Data\*.csv"):
    n = sum(1 for _ in open(f, encoding="utf-8", errors="ignore")) - 1
    print(f"{os.path.basename(f):45s} {n:>12,} rows")

allcars.csv                                      7,214,695 rows
CAR_Prediction_System_DATASET.csv                   19,473 rows
database.csv                                        38,113 rows
engine_data.csv                                     19,535 rows
exp1_14drivers_14cars_dailyRoutes.csv               60,439 rows
exp2_19drivers_1car_1route.csv                       8,261 rows
exp3_4drivers_1car_1route.csv                        1,743 rows
measurements.csv                                       388 rows
MY2022 Fuel Consumption Ratings.csv                    946 rows


In [19]:
import requests, pandas as pd

r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 42.28, "longitude": -83.74,
    "start_date": "2017-11-01", "end_date": "2018-11-30",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
    "timezone": "America/Detroit",
}, timeout=60)
r.raise_for_status()
weather = pd.DataFrame(r.json()["hourly"])
weather["time"] = pd.to_datetime(weather["time"])
weather.to_csv(r"D:\Ml project\Data\weather_annarbor.csv", index=False)
print(weather.shape)

ModuleNotFoundError: No module named 'requests'

In [20]:
%pip install requests

  Using cached requests-2.34.2-py3-none-any.whl.metadata (4.8 kB)
  Using cached idna-3.20-py3-none-any.whl.metadata (7.2 kB)
  Using cached urllib3-2.8.0-py3-none-any.whl.metadata (7.4 kB)
Using cached requests-2.34.2-py3-none-any.whl (73 kB)
Using cached idna-3.20-py3-none-any.whl (69 kB)
Using cached urllib3-2.8.0-py3-none-any.whl (135 kB)

   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   ---------------------------------------- 0/5 [urllib3]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   -------- ------------------------------- 1/5 [idna]
   ---------------- -----


[notice] A new release of pip is available: 26.1.2 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [21]:
import requests, pandas as pd

r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 42.28, "longitude": -83.74,
    "start_date": "2017-11-01", "end_date": "2018-11-30",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
    "timezone": "America/Detroit",
}, timeout=60)
r.raise_for_status()
weather = pd.DataFrame(r.json()["hourly"])
weather["time"] = pd.to_datetime(weather["time"])
weather.to_csv(r"D:\Ml project\Data\weather_annarbor.csv", index=False)
print(weather.shape)
print(weather.head())

(9480, 5)
                 time  temperature_2m  relative_humidity_2m  precipitation  \
0 2017-11-01 00:00:00             2.0                    68            0.0   
1 2017-11-01 01:00:00             1.4                    71            0.0   
2 2017-11-01 02:00:00             1.4                    69            0.0   
3 2017-11-01 03:00:00             1.0                    72            0.0   
4 2017-11-01 04:00:00             0.9                    74            0.0   

   wind_speed_10m  
0             4.7  
1             6.7  
2             7.2  
3             6.2  
4             8.1  


In [22]:
r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 42.28, "longitude": -83.74,
    "start_date": "2017-10-01", "end_date": "2018-12-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
    "timezone": "America/Detroit",
}, timeout=60)
r.raise_for_status()
weather = pd.DataFrame(r.json()["hourly"])
weather["time"] = pd.to_datetime(weather["time"])
weather.to_csv(r"D:\Ml project\Data\weather_annarbor.csv", index=False)
print(weather.shape)   # expect (10968, 5)

(10968, 5)


In [2]:
import pandas as pd, numpy as np
from pathlib import Path

root = Path("Data")
v = pd.read_csv(root / "VED_DynamicData_Part1" / "VED_171108_week.csv", low_memory=False)
print(root.resolve(), v.shape)

D:\Ml project\Data (535198, 22)


In [23]:
import glob, pandas as pd
f = glob.glob(r"D:\Ml project\Data\VED_DynamicData_Part1\*.csv")[0]
v = pd.read_csv(f, nrows=3)
print(f)
print(v.columns.tolist())
print(v)

D:\Ml project\Data\VED_DynamicData_Part1\VED_171101_week.csv
['DayNum', 'VehId', 'Trip', 'Timestamp(ms)', 'Latitude[deg]', 'Longitude[deg]', 'Vehicle Speed[km/h]', 'MAF[g/sec]', 'Engine RPM[RPM]', 'Absolute Load[%]', 'OAT[DegC]', 'Fuel Rate[L/hr]', 'Air Conditioning Power[kW]', 'Air Conditioning Power[Watts]', 'Heater Power[Watts]', 'HV Battery Current[A]', 'HV Battery SOC[%]', 'HV Battery Voltage[V]', 'Short Term Fuel Trim Bank 1[%]', 'Short Term Fuel Trim Bank 2[%]', 'Long Term Fuel Trim Bank 1[%]', 'Long Term Fuel Trim Bank 2[%]']
     DayNum  VehId  Trip  Timestamp(ms)  Latitude[deg]  Longitude[deg]  \
0  1.586651      8   706              0      42.277558       -83.69875   
1  1.586651      8   706            200      42.277558       -83.69875   
2  1.586651      8   706           1100      42.277558       -83.69875   

   Vehicle Speed[km/h]  MAF[g/sec]  Engine RPM[RPM]  Absolute Load[%]  ...  \
0                   40   22.129999             2285         49.019608  ...   
1      

In [24]:
import pandas as pd

weather = pd.read_csv(r"D:\Ml project\Data\weather_annarbor.csv", parse_dates=["time"])

ved = pd.read_csv(r"D:\Ml project\Data\VED_DynamicData_Part1\VED_171101_week.csv")

start = pd.Timestamp("2017-11-01")
ved["datetime"] = start + pd.to_timedelta(ved["DayNum"] - 1, unit="D") \
                        + pd.to_timedelta(ved["Timestamp(ms)"], unit="ms")
ved["hour"] = ved["datetime"].dt.floor("h")

merged = ved.merge(weather, left_on="hour", right_on="time", how="left")

print("rows:", len(merged))
print("unmatched weather:", merged["temperature_2m"].isna().mean())
print(merged[["OAT[DegC]", "temperature_2m"]].corr())
print(ved["datetime"].min(), "->", ved["datetime"].max())

rows: 489414
unmatched weather: 0.0
                OAT[DegC]  temperature_2m
OAT[DegC]        1.000000       -0.072918
temperature_2m  -0.072918        1.000000
2017-11-01 00:04:13.838400480 -> 2017-11-08 00:03:48.266399776


In [25]:
import numpy as np

# 1. Is OAT sane?
print(ved["OAT[DegC]"].describe())
print(ved["OAT[DegC]"].notna().mean(), "share non-null")

# 2. Clean it and average per hour, to cut sensor noise
v = ved[ved["OAT[DegC]"].between(-30, 45)].copy()
hourly = v.groupby("hour")["OAT[DegC]"].mean().rename("oat").to_frame()

# 3. Try time shifts (hours) on the weather side, to find the best alignment
w = weather.set_index("time")["temperature_2m"]
for shift in range(-24, 25, 4):
    j = hourly.join(w.shift(shift, freq="h"), how="inner")
    print(f"shift {shift:+3d}h  corr = {j['oat'].corr(j['temperature_2m']):.3f}  n={len(j)}")

count    55169.000000
mean         8.334672
std          4.886611
min          1.500000
25%          5.000000
50%          7.500000
75%         10.000000
max         30.000000
Name: OAT[DegC], dtype: float64
0.1127246053443506 share non-null
shift -24h  corr = -0.152  n=59
shift -20h  corr = 0.204  n=59
shift -16h  corr = 0.130  n=59
shift -12h  corr = -0.088  n=59
shift  -8h  corr = -0.105  n=59
shift  -4h  corr = -0.137  n=59
shift  +0h  corr = -0.114  n=59
shift  +4h  corr = 0.067  n=59
shift  +8h  corr = 0.011  n=59
shift +12h  corr = -0.196  n=59
shift +16h  corr = -0.328  n=59
shift +20h  corr = -0.398  n=59
shift +24h  corr = -0.352  n=59


In [26]:
trip_start = ved.groupby(["VehId", "Trip"])["datetime"].min()
print(trip_start.dt.hour.value_counts().sort_index())

datetime
0     37
1     26
2     15
3      9
4      4
5      3
6      2
7      2
9      3
10     4
11    21
12    59
13    70
14    53
15    39
16    62
17    69
18    64
19    52
20    57
21    77
22    76
23    50
Name: count, dtype: int64


In [27]:
import requests, pandas as pd

r = requests.get("https://archive-api.open-meteo.com/v1/archive", params={
    "latitude": 42.28, "longitude": -83.74,
    "start_date": "2017-10-01", "end_date": "2018-12-31",
    "hourly": "temperature_2m,relative_humidity_2m,precipitation,wind_speed_10m",
    "timezone": "UTC",
}, timeout=60)
r.raise_for_status()
weather = pd.DataFrame(r.json()["hourly"])
weather["time"] = pd.to_datetime(weather["time"])
weather.to_csv(r"D:\Ml project\Data\weather_annarbor_utc.csv", index=False)
print(weather.shape)   # (10968, 5)

(10968, 5)


In [28]:
ved["datetime"] = pd.Timestamp("2017-11-01") + pd.to_timedelta(ved["DayNum"] - 1, unit="D") \
                  + pd.to_timedelta(ved["Timestamp(ms)"], unit="ms")
ved["hour"] = ved["datetime"].dt.floor("h")
merged = ved.merge(weather, left_on="hour", right_on="time", how="left")

# Confirm the time mapping: convert UTC to Detroit and re-check trip starts
ts = ved.groupby(["VehId","Trip"])["datetime"].min()
local = ts.dt.tz_localize("UTC").dt.tz_convert("America/Detroit")
print(local.dt.hour.value_counts().sort_index())

# Daily OAT vs API temperature
v = ved[ved["OAT[DegC]"].between(-30, 45)].copy()
d_oat = v.groupby(v["datetime"].dt.floor("D"))["OAT[DegC]"].mean()
w_day = weather.set_index("time")["temperature_2m"].resample("D").mean()
j = pd.concat([d_oat, w_day], axis=1, join="inner")
print(j); print(j.corr())

datetime
0      3
1      2
2      2
3      2
5      5
6      6
7     33
8     63
9     70
10    47
11    51
12    64
13    68
14    62
15    47
16    59
17    76
18    71
19    45
20    34
21    23
22    16
23     5
Name: count, dtype: int64
            OAT[DegC]  temperature_2m
2017-11-01   8.366058        3.291667
2017-11-02   4.283063        9.795833
2017-11-03   4.561552        8.895833
2017-11-04  12.753643        5.787500
2017-11-05  11.565534       12.716667
2017-11-06   6.780318        7.058333
2017-11-07  10.354061        4.012500
2017-11-08  12.000000        2.404167
                OAT[DegC]  temperature_2m
OAT[DegC]        1.000000       -0.328435
temperature_2m  -0.328435        1.000000


In [29]:
import glob, pandas as pd

parts = []
for f in sorted(glob.glob(r"D:\Ml project\Data\VED_DynamicData_Part*\*.csv")):
    d = pd.read_csv(f, usecols=["DayNum", "Timestamp(ms)", "OAT[DegC]"]).dropna(subset=["OAT[DegC]"])
    d = d[d["OAT[DegC]"].between(-30, 45)]
    d["hour"] = (pd.Timestamp("2017-11-01")
                 + pd.to_timedelta(d["DayNum"] - 1, unit="D")
                 + pd.to_timedelta(d["Timestamp(ms)"], unit="ms")).dt.floor("h")
    parts.append(d.groupby("hour")["OAT[DegC]"].agg(["sum", "count"]))

agg = pd.concat(parts).groupby(level=0).sum()
oat = (agg["sum"] / agg["count"]).rename("oat")

w = pd.read_csv(r"D:\Ml project\Data\weather_annarbor_utc.csv", parse_dates=["time"]).set_index("time")["temperature_2m"]

for shift in [-12, -6, -5, -4, 0, 4, 5, 6, 12]:
    j = pd.concat([oat, w.shift(shift, freq="h")], axis=1, join="inner").dropna()
    print(f"shift {shift:+3d}h  corr = {j.iloc[:,0].corr(j.iloc[:,1]):.3f}  n={len(j)}")

shift -12h  corr = 0.798  n=5502
shift  -6h  corr = 0.816  n=5502
shift  -5h  corr = 0.830  n=5502
shift  -4h  corr = 0.844  n=5502
shift  +0h  corr = 0.883  n=5502
shift  +4h  corr = 0.862  n=5502
shift  +5h  corr = 0.853  n=5502
shift  +6h  corr = 0.846  n=5502
shift +12h  corr = 0.821  n=5502


In [30]:
import pandas as pd, numpy as np, time
from pathlib import Path
root = Path("Data")

VA = pd.read_pickle(root / "ved_features.pkl")
N = 8000
VA["_r"] = VA.groupby("VehId").cumcount()
VA["_n"] = VA.groupby("VehId")["dt"].transform("size")
S = VA[VA["_r"] % (VA["_n"] // N).clip(lower=1) == 0]
S = S[S.groupby("VehId").cumcount() < N].drop(columns=["_r", "_n"]).reset_index(drop=True)
del VA

S["y"] = S["maf"] / 14.7 * 3600 / 750          # fuel, L/h
S["L_true"] = S["y"] * S["dt"] / 3600
S["km"] = S["speed"] * S["dt"] / 3600
FEATS = ["rpm", "speed", "speed_sq", "rpm_per_kmh", "is_idle", "is_highway", "accel",
         "accel_abs", "rpm_change", "speed_roll5", "rpm_roll5", "speed_std5", "disp_L"]
print(S.shape, "cars:", S["VehId"].nunique())
print(S.groupby("VehId").size().describe().round(0))

(1359929, 19) cars: 196
count     196.0
mean     6938.0
std      2326.0
min        71.0
25%      8000.0
50%      8000.0
75%      8000.0
max      8000.0
dtype: float64


In [ ]:
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

models = {
    "baseline_mean": lambda: DummyRegressor(),
    "ridge": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge()),
    "random_forest": lambda: make_pipeline(SimpleImputer(strategy="median"),
    RandomForestRegressor(n_estimators=40, min_samples_leaf=30, max_samples=0.1, n_jobs=-1, random_state=0)
    "hist_gb": lambda: HistGradientBoostingRegressor(random_state=0),
}
rows, car_err = [], {}
for name, mk in models.items():
    pred = np.zeros(len(S)); t0 = time.time()
    for tr, te in GroupKFold(5).split(S, groups=S["VehId"]):
        m = mk().fit(S.iloc[tr][FEATS], S.iloc[tr]["y"])
        pred[te] = m.predict(S.iloc[te][FEATS])
    secs = time.time() - t0
    T = S[["VehId", "L_true", "km"]].assign(L_pred=pred * S["dt"] / 3600).groupby("VehId").sum()
    T = T[T["km"] >= 10]
    e = 100 * (T["L_pred"] - T["L_true"]).abs() / T["L_true"]
    car_err[name] = e
    rows.append({"model": name, "MAE_Lh": np.abs(pred - S["y"]).mean(), "cars_scored": len(e),
                 "median_err_%": e.median(), "p75_%": e.quantile(.75), "p90_%": e.quantile(.9),
                 "train_secs": secs})
print(pd.DataFrame(rows).round(2).to_string(index=False))
}

SyntaxError: closing parenthesis '}' does not match opening parenthesis '(' on line 12 (950240696.py, line 15)

In [1]:
import pandas as pd
VA = pd.read_pickle("Data/ved_features.pkl")
print(VA[["is_idle","is_highway"]].describe())
print("speed==0:", (VA.speed==0).mean(), " speed>80:", (VA.speed>80).mean())
print(VA.groupby("VehId").maf.mean().describe())

            is_idle    is_highway
count  1.020302e+07  1.020302e+07
mean   1.535661e-01  5.451554e-02
std    3.605324e-01  2.270322e-01
min    0.000000e+00  0.000000e+00
25%    0.000000e+00  0.000000e+00
50%    0.000000e+00  0.000000e+00
75%    0.000000e+00  0.000000e+00
max    1.000000e+00  1.000000e+00
speed==0: 0.14563583049642762  speed>80: 0.05451553648146068
count    196.000000
mean      12.361687
std        4.237591
min        6.554029
25%        9.932657
50%       11.616166
75%       13.604213
max       40.959999
Name: maf, dtype: float64


In [2]:
g = VA.groupby("VehId").maf.agg(["mean","count"]).sort_values("mean", ascending=False)
print(g.head(5))
print(VA.loc[VA.is_idle==1,"speed"].describe())
static = pd.read_excel("Data/VED_Static_Data_ICE&HEV.xlsx")
print(static.columns.tolist())
print(static.iloc[:,:6].head())

            mean  count
VehId                  
252    40.959999     71
217    35.435272   5238
505    29.183283    393
131    26.673260   2261
108    22.805132  18295
count    1.566838e+06
mean     5.164095e-02
std      2.213011e-01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      0.000000e+00
max      1.000000e+00
Name: speed, dtype: float64
['VehId', 'Vehicle Type', 'Vehicle Class', 'Engine Configuration & Displacement', 'Transmission', 'Drive Wheels', 'Generalized_Weight']
   VehId Vehicle Type Vehicle Class Engine Configuration & Displacement  \
0      2          ICE           Car                       4-FI 2.0L T/C   
1      5          HEV           Car                 4-GAS/ELECTRIC 2.0L   
2      7          ICE           SUV                           6-FI 3.6L   
3      8          ICE           Car                           4-FI 1.5L   
4     12          ICE           Car                           4-FI 1.8L   

  Transmission Drive Wheels  
0      NO D

In [32]:
import time
import numpy as np
import pandas as pd
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor

forest = RandomForestRegressor(n_estimators=40, min_samples_leaf=30,
                               max_samples=0.1, n_jobs=-1, random_state=0)

models = {
    "baseline_mean": lambda: DummyRegressor(),
    "ridge": lambda: make_pipeline(SimpleImputer(strategy="median"), StandardScaler(), Ridge()),
    "random_forest": lambda: make_pipeline(SimpleImputer(strategy="median"), forest),
    "hist_gb": lambda: HistGradientBoostingRegressor(random_state=0),
}

rows, car_err = [], {}
for name, mk in models.items():
    pred = np.zeros(len(S))
    t0 = time.time()
    for tr, te in GroupKFold(5).split(S, groups=S["VehId"]):
        m = mk().fit(S.iloc[tr][FEATS], S.iloc[tr]["y"])
        pred[te] = m.predict(S.iloc[te][FEATS])
    secs = time.time() - t0
    T = S[["VehId", "L_true", "km"]].assign(L_pred=pred * S["dt"] / 3600).groupby("VehId").sum()
    T = T[T["km"] >= 10]
    e = 100 * (T["L_pred"] - T["L_true"]).abs() / T["L_true"]
    car_err[name] = e
    rows.append({"model": name, "MAE_Lh": np.abs(pred - S["y"]).mean(),
                 "cars_scored": len(e), "median_err_%": e.median(),
                 "p75_%": e.quantile(.75), "p90_%": e.quantile(.9),
                 "train_secs": secs})
    print(name, "done in", round(secs), "s | median per-car error", round(e.median(), 1), "%")

print(pd.DataFrame(rows).round(2).to_string(index=False))

baseline_mean done in 4 s | median per-car error 19.2 %
ridge done in 33 s | median per-car error 8.4 %
random_forest done in 125 s | median per-car error 7.3 %
hist_gb done in 57 s | median per-car error 7.4 %
        model  MAE_Lh  cars_scored  median_err_%  p75_%  p90_%  train_secs
baseline_mean    2.82          182         19.19  34.27  53.55        4.44
        ridge    1.54          182          8.45  15.00  24.32       33.04
random_forest    1.23          182          7.30  13.19  20.10      124.98
      hist_gb    1.23          182          7.40  13.33  20.62       57.08


In [3]:
a = pd.read_csv(root / "allcars.csv", low_memory=False, nrows=200000)
print(a.shape, "(first 200k rows)")
print(list(a.columns))
print(a.head(3).T.to_string())
print(a.notna().mean().round(2).to_string())

(200000, 18) (first 200k rows)
['Unnamed: 0', 'timeStamp', 'tripID', 'accData', 'gps_speed', 'battery', 'cTemp', 'dtc', 'eLoad', 'iat', 'imap', 'kpl', 'maf', 'rpm', 'speed', 'tAdv', 'tPos', 'deviceID']
                                                                                                                                                                             0                                                                                                                                                                   1                                                                                                                                                                   2
Unnamed: 0                                                                                                                                                                   0                                                                                                                         

In [34]:
from sklearn.inspection import permutation_importance
from sklearn.feature_selection import mutual_info_regression
from scipy.stats import spearmanr, wilcoxon

def car_errors(feats):
    pred = np.zeros(len(S))
    for tr, te in GroupKFold(5).split(S, groups=S["VehId"]):
        m = HistGradientBoostingRegressor(random_state=0).fit(S.iloc[tr][feats], S.iloc[tr]["y"])
        pred[te] = m.predict(S.iloc[te][feats])
    T = S[["VehId", "L_true", "km"]].assign(L_pred=pred * S["dt"] / 3600).groupby("VehId").sum()
    T = T[T["km"] >= 10]
    return 100 * (T["L_pred"] - T["L_true"]).abs() / T["L_true"], np.abs(pred - S["y"]).mean()

no_eng = [f for f in FEATS if f != "disp_L"]
abl = {}
for label, feats in [("all features", FEATS), ("without disp_L", no_eng), ("disp_L only", ["disp_L"])]:
    e, mae = car_errors(feats)
    abl[label] = e
    print(f"{label:15s} median {e.median():5.2f}% | p75 {e.quantile(.75):5.2f}% | p90 {e.quantile(.9):5.2f}% | MAE {mae:.2f} L/h")

for a in ["without disp_L", "disp_L only"]:
    d = (abl[a] - abl["all features"]).dropna()
    print(f"{a} vs all features: median diff {d.median():.2f} pts, Wilcoxon p = {wilcoxon(d)[1]:.3g}")

# --- feature selection by two methods ---
tr, te = next(GroupKFold(5).split(S, groups=S["VehId"]))
m = HistGradientBoostingRegressor(random_state=0).fit(S.iloc[tr][FEATS], S.iloc[tr]["y"])
sub = S.iloc[te].sample(min(60000, len(te)), random_state=0)
pi = permutation_importance(m, sub[FEATS], sub["y"], n_repeats=3, random_state=0,
                            scoring="neg_mean_absolute_error")
sm = S[FEATS].sample(100000, random_state=0)
mi = mutual_info_regression(sm.fillna(sm.median()), S.loc[sm.index, "y"], random_state=0)
tab = pd.DataFrame({"perm_MAE_increase": pi.importances_mean, "mutual_info": mi}, index=FEATS)
tab["rank_perm"] = tab["perm_MAE_increase"].rank(ascending=False)
tab["rank_mi"] = tab["mutual_info"].rank(ascending=False)
print(tab.sort_values("rank_perm").round(3))
print("Spearman rank agreement:", round(spearmanr(tab["rank_perm"], tab["rank_mi"])[0], 2))

all features    median  7.40% | p75 13.33% | p90 20.62% | MAE 1.23 L/h
without disp_L  median 18.34% | p75 33.58% | p90 45.43% | MAE 1.48 L/h
disp_L only     median 13.07% | p75 23.87% | p90 39.81% | MAE 2.86 L/h
without disp_L vs all features: median diff 8.62 pts, Wilcoxon p = 3.21e-14
disp_L only vs all features: median diff 4.36 pts, Wilcoxon p = 3.42e-08
             perm_MAE_increase  mutual_info  rank_perm  rank_mi
rpm                      1.930        0.488        1.0      1.0
disp_L                   0.612        0.371        2.0      3.0
speed                    0.397        0.276        3.0      4.0
rpm_per_kmh              0.383        0.198        4.0      6.0
accel                    0.151        0.147        5.0      9.0
speed_roll5              0.066        0.172        6.0      7.0
speed_std5               0.058        0.159        7.0      8.0
rpm_roll5                0.031        0.386        8.0      2.0
rpm_change               0.016        0.052        9.0     13.

In [35]:
sets = {
    "9 selected": ["rpm", "disp_L", "speed", "rpm_per_kmh", "accel",
                   "speed_roll5", "speed_std5", "rpm_roll5", "rpm_change"],
    "5 minimal": ["rpm", "disp_L", "speed", "rpm_per_kmh", "accel"],
}
for label, feats in sets.items():
    e, mae = car_errors(feats)
    abl[label] = e
    d = (e - abl["all features"]).dropna()
    print(f"{label:11s} ({len(feats)} features) median {e.median():5.2f}% | p75 {e.quantile(.75):5.2f}% | "
          f"p90 {e.quantile(.9):5.2f}% | MAE {mae:.2f} L/h | vs all 13: diff {d.median():+.2f} pts, p = {wilcoxon(d)[1]:.3g}")

9 selected  (9 features) median  7.56% | p75 13.34% | p90 20.51% | MAE 1.23 L/h | vs all 13: diff -0.02 pts, p = 0.118
5 minimal   (5 features) median  7.41% | p75 14.65% | p90 21.66% | MAE 1.28 L/h | vs all 13: diff +0.47 pts, p = 1.96e-05


In [36]:
from sklearn.ensemble import HistGradientBoostingRegressor
F9 = ["rpm", "disp_L", "speed", "rpm_per_kmh", "accel",
      "speed_roll5", "speed_std5", "rpm_roll5", "rpm_change"]

def numc(s):
    return pd.to_numeric(s.astype(str).str.replace(",", ".", regex=False), errors="coerce")

raw = pd.read_csv(root / "exp1_14drivers_14cars_dailyRoutes.csv", low_memory=False, dtype=str)
X = raw[raw["VEHICLE_ID"].isin(["car1", "car11", "car12"])].copy()
for c in ["MAF", "ENGINE_RPM", "SPEED", "ENGINE_POWER", "TIMESTAMP"]:
    X[c] = numc(X[c])
X = X.rename(columns={"MAF": "maf", "ENGINE_RPM": "rpm", "SPEED": "speed", "ENGINE_POWER": "disp_L"})
X = X.dropna(subset=["rpm", "speed", "TIMESTAMP"]).sort_values(["VEHICLE_ID", "TIMESTAMP"])

g = X.groupby("VEHICLE_ID")
X["dt"] = g["TIMESTAMP"].diff() / 1000
X["rpm_per_kmh"] = (X["rpm"] / X["speed"].clip(lower=1)).clip(upper=500)
X["accel"] = (g["speed"].diff() / X["dt"]).clip(-10, 10)
X["rpm_change"] = g["rpm"].diff().abs()
X["speed_roll5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=1).mean())
X["speed_std5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=2).std())
X["rpm_roll5"] = g["rpm"].transform(lambda z: z.rolling(5, min_periods=1).mean())
X = X[X["dt"].between(0.1, 30) & X["maf"].between(0.5, 250)].copy()

m9 = HistGradientBoostingRegressor(random_state=0).fit(S[F9], S["maf"])
X["pred"] = m9.predict(X[F9])
c = X.assign(a=X["maf"] * X["dt"], p=X["pred"] * X["dt"]).groupby("VEHICLE_ID")[["a", "p"]].sum()
c["signed_err_%"] = 100 * (c["p"] / c["a"] - 1)
c["rows"] = X.groupby("VEHICLE_ID").size()
c["row_MAE_gs"] = (X["pred"] - X["maf"]).abs().groupby(X["VEHICLE_ID"]).mean()
print(c.round(1))

c["ratio"] = c["p"] / c["a"]
for car in c.index:
    k = c.drop(car)["ratio"].mean()
    print(car, "error after calibrating on the other two:", round(100 * (c.loc[car, "ratio"] / k - 1), 1), "%")

                   a          p  signed_err_%  rows  row_MAE_gs
VEHICLE_ID                                                     
car1        168193.9   253416.2          50.7  2669         4.7
car11       893709.8  1092775.5          22.3  7210         5.1
car12       114337.3   154363.7          35.0  1603         4.6
car1 error after calibrating on the other two: 17.1 %
car11 error after calibrating on the other two: -14.4 %
car12 error after calibrating on the other two: -1.1 %


In [37]:
F8 = [f for f in F9 if f != "disp_L"]
keep = [4, 12, 13, 23, 43, 45, 46, 47, 53]
use = ["timeStamp", "tripID", "speed", "maf", "rpm", "deviceID"]
A = pd.read_csv(root / "allcars.csv", usecols=use, low_memory=False)
A = A[A["deviceID"].isin(keep)].copy()
A["t"] = pd.to_datetime(A["timeStamp"], errors="coerce")
A = A.sort_values(["deviceID", "tripID", "t"])

k = ["deviceID", "tripID"]
g = A.groupby(k)
A["dt"] = g["t"].diff().dt.total_seconds()
A["rpm_per_kmh"] = (A["rpm"] / A["speed"].clip(lower=1)).clip(upper=500)
A["accel"] = (g["speed"].diff() / A["dt"]).clip(-10, 10)
A["rpm_change"] = g["rpm"].diff().abs()
A["speed_roll5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=1).mean())
A["speed_std5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=2).std())
A["rpm_roll5"] = g["rpm"].transform(lambda z: z.rolling(5, min_periods=1).mean())
A = A[A["dt"].between(0.1, 30) & A["maf"].between(0.5, 250) & A["speed"].notna()].copy()

m8 = HistGradientBoostingRegressor(random_state=0).fit(S[F8], S["maf"])
A["pred"] = m8.predict(A[F8])
c2 = A.assign(a=A["maf"] * A["dt"], p=A["pred"] * A["dt"]).groupby("deviceID")[["a", "p"]].sum()
c2["signed_err_%"] = 100 * (c2["p"] / c2["a"] - 1)
c2["rows"] = A.groupby("deviceID").size()
c2["row_MAE_gs"] = (A["pred"] - A["maf"]).abs().groupby(A["deviceID"]).mean()
print(c2.round(1))
print("median absolute device error:", round(c2["signed_err_%"].abs().median(), 1), "%")

                  a          p  signed_err_%    rows  row_MAE_gs
deviceID                                                        
4         5259083.4  6197158.6          17.8  834768         1.3
12        1274169.8  2531315.4          98.7  152200         8.6
13        2104344.3  1370067.4         -34.9  137036         5.6
23        5439441.7  3840788.3         -29.4  423808         4.0
43        1172212.4  2408139.8         105.4  204842         6.2
45        5495332.6  3808400.1         -30.7  437516         4.2
46        4031047.9  2818162.5         -30.1  176911         6.8
47         860276.0   597187.0         -30.6   71606         4.2
53        8400886.4  5390686.2         -35.8  616685         5.1
median absolute device error: 30.7 %


In [38]:
dev = A.groupby("deviceID").agg(maf_med=("maf", "median"), rpm_med=("rpm", "median"),
                                speed_med=("speed", "median")).join(c2["signed_err_%"])
print(dev.sort_values("maf_med").round(1))
print("corr(median MAF, signed error):", round(dev["maf_med"].corr(dev["signed_err_%"]), 2))

base = S["maf"].mean()
cb = A.assign(a=A["maf"] * A["dt"], p=base * A["dt"]).groupby("deviceID")[["a", "p"]].sum()
print("constant-guess median absolute error:", round((100 * (cb["p"] / cb["a"] - 1)).abs().median(), 1), "%")

          maf_med  rpm_med  speed_med  signed_err_%
deviceID                                           
43            3.8   1389.0       26.0         105.4
12            5.4   1664.0       33.0          98.7
4             6.0   1200.0       36.0          17.8
47            9.4   1060.0       21.0         -30.6
45           10.5   1147.0       29.0         -30.7
53           11.4   1089.5       20.0         -35.8
13           11.9   1175.0       34.0         -34.9
23           13.1   1320.5       66.0         -29.4
46           23.1   1816.5       55.0         -30.1
corr(median MAF, signed error): -0.67
constant-guess median absolute error: 21.2 %


In [39]:
def ve_w(d):
    return d["maf"].sum() / (d["disp_L"] * d["rpm"] / 120 * 1.2).sum()

ve0 = ve_w(S)
air = A.groupby("deviceID").apply(lambda d: (d["rpm"] / 120 * 1.2).sum())
imp = A.groupby("deviceID")["maf"].sum() / (ve0 * air)
print("implied displacement (L):", imp.round(2).to_dict())

A["disp_L"] = A["deviceID"].map(imp)
A["pred9"] = m9.predict(A[F9])
c3 = A.assign(a=A["maf"] * A["dt"], p=A["pred9"] * A["dt"]).groupby("deviceID")[["a", "p"]].sum()
c3["signed_err_%"] = (100 * (c3["p"] / c3["a"] - 1)).round(1)
print(c3[["signed_err_%"]])
print("median absolute error:", round(c3["signed_err_%"].abs().median(), 1), "%")

implied displacement (L): {4: 1.58, 12: 1.42, 13: 3.67, 23: 3.17, 43: 1.2, 45: 3.23, 46: 3.85, 47: 3.11, 53: 3.47}
          signed_err_%
deviceID              
4                -20.7
12                51.3
13               -12.7
23               -18.4
43                49.5
45               -21.4
46                 6.4
47               -19.2
53               -22.6
median absolute error: 20.7 %


In [40]:
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
eng = st["Engine Configuration & Displacement"].astype(str)
st["cyl"] = pd.to_numeric(eng.str.extract(r"^(\d+)")[0], errors="coerce")
st["turbo"] = eng.str.contains("T/C|SC", regex=True).astype(int)
st["wt"] = pd.to_numeric(st["Generalized_Weight"], errors="coerce")

S2 = S.merge(st[["VehId", "cyl", "turbo", "wt"]], on="VehId", how="left")
S2 = S2[(S2["rpm"] > 300) & (S2["disp_L"] > 0)].reset_index(drop=True)
S2["air_cap"] = S2["disp_L"] * S2["rpm"] / 120 * 1.2
S2["ve"] = (S2["maf"] / S2["air_cap"]).clip(0, 3)

F_ENG = F9 + ["cyl", "turbo", "wt"]

def run(feats, target):
    pred = np.zeros(len(S2))
    for tr, te in GroupKFold(5).split(S2, groups=S2["VehId"]):
        col = "y" if target == "y" else "ve"
        m = HistGradientBoostingRegressor(random_state=0).fit(S2.iloc[tr][feats], S2.iloc[tr][col])
        p = m.predict(S2.iloc[te][feats])
        pred[te] = p if target == "y" else p * S2.iloc[te]["air_cap"].values * 3600 / 14.7 / 750
    T = S2[["VehId", "L_true", "km"]].assign(L_pred=pred * S2["dt"] / 3600).groupby("VehId").sum()
    T = T[T["km"] >= 10]
    return 100 * (T["L_pred"] - T["L_true"]).abs() / T["L_true"], np.abs(pred - S2["y"]).mean()

res = {}
for label, feats, tgt in [("A: 9 feats, fuel target", F9, "y"),
                          ("B: 9 feats, VE target", F9, "ve"),
                          ("C: +cyl/turbo/weight, fuel", F_ENG, "y"),
                          ("D: +cyl/turbo/weight, VE", F_ENG, "ve")]:
    e, mae = run(feats, tgt)
    res[label] = e
    d = (e - res["A: 9 feats, fuel target"]).dropna()
    p = wilcoxon(d)[1] if len(d) and d.abs().sum() > 0 else float("nan")
    print(f"{label:28s} median {e.median():5.2f}% | p75 {e.quantile(.75):5.2f}% | p90 {e.quantile(.9):5.2f}% "
          f"| MAE {mae:.2f} L/h | vs A: {d.median():+.2f} pts, p = {p:.3g}")

# ---- transfer to exp1: fuel target vs VE target ----
X["air_cap"] = X["disp_L"] * X["rpm"] / 120 * 1.2
for label, tgt in [("fuel target", "maf"), ("VE target", "ve")]:
    mm = HistGradientBoostingRegressor(random_state=0).fit(S2[F9], S2[tgt])
    p = mm.predict(X[F9])
    pred_maf = p if tgt == "maf" else p * X["air_cap"].values
    c = X.assign(a=X["maf"] * X["dt"], p=pred_maf * X["dt"]).groupby("VEHICLE_ID")[["a", "p"]].sum()
    print(f"exp1 {label:12s} signed error %:", (100 * (c["p"] / c["a"] - 1)).round(1).to_dict())

A: 9 feats, fuel target      median  6.92% | p75 13.19% | p90 20.36% | MAE 1.23 L/h | vs A: +0.00 pts, p = nan
B: 9 feats, VE target        median  6.64% | p75 13.31% | p90 21.29% | MAE 1.21 L/h | vs A: -0.22 pts, p = 0.0856
C: +cyl/turbo/weight, fuel   median  7.12% | p75 13.56% | p90 19.24% | MAE 1.23 L/h | vs A: -0.07 pts, p = 0.969
D: +cyl/turbo/weight, VE     median  6.77% | p75 12.44% | p90 20.31% | MAE 1.21 L/h | vs A: -0.20 pts, p = 0.272
exp1 fuel target  signed error %: {'car1': 50.6, 'car11': 23.9, 'car12': 32.3}
exp1 VE target    signed error %: {'car1': 59.6, 'car11': 21.8, 'car12': 27.5}


In [41]:
import requests, time

def call(label, url, **params):
    t0 = time.time()
    try:
        r = requests.get(url, params=params, headers={"Accept": "application/json"}, timeout=25)
        ms = round(1000 * (time.time() - t0))
        print(f"{label:34s} status {r.status_code} | {ms} ms")
        return r.json() if r.ok else None
    except Exception as e:
        print(f"{label:34s} FAILED: {type(e).__name__}: {str(e)[:90]}")
        return None

# 1. NHTSA vPIC: decode a VIN into engine specs
j = call("NHTSA vPIC VIN decode",
         "https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/1HGCM82633A004352", format="json")
if j:
    d = j["Results"][0]
    print("   ", {k: d.get(k) for k in ["Make", "Model", "ModelYear", "EngineCylinders",
                                        "DisplacementL", "FuelTypePrimary", "ErrorCode"]})

# 2. NHTSA recalls for a car in your data, then one that is not sold in the US
j = call("NHTSA recalls: Corolla 2009", "https://api.nhtsa.gov/recalls/recallsByVehicle",
         make="toyota", model="corolla", modelYear=2009)
if j:
    print("    recalls:", j.get("Count"), "| first component:",
          (j.get("results") or [{}])[0].get("Component"))
j = call("NHTSA recalls: Suzuki Alto 2019", "https://api.nhtsa.gov/recalls/recallsByVehicle",
         make="suzuki", model="alto", modelYear=2019)
if j:
    print("    recalls:", j.get("Count"), "(expect 0 or few; not a US-market model)")

# 3. NHTSA complaints
j = call("NHTSA complaints: Corolla 2009", "https://api.nhtsa.gov/complaints/complaintsByVehicle",
         make="toyota", model="corolla", modelYear=2009)
if j:
    print("    complaints:", j.get("count", j.get("Count")))

# 4. Open-Meteo weather (Islamabad)
j = call("Open-Meteo current weather", "https://api.open-meteo.com/v1/forecast",
         latitude=33.68, longitude=73.05, current="temperature_2m,relative_humidity_2m")
if j:
    print("   ", j.get("current"))

# 5. fueleconomy.gov web services
j = call("fueleconomy.gov models 2009", "https://www.fueleconomy.gov/ws/rest/vehicle/menu/model",
         year=2009, make="Toyota")
if j:
    print("    first models:", [m.get("text") for m in j.get("menuItem", [])][:5])
j = call("fueleconomy.gov fuel prices", "https://www.fueleconomy.gov/ws/rest/fuelprices")
if j:
    print("   ", {k: j[k] for k in list(j)[:5]})

NHTSA vPIC VIN decode              status 200 | 1358 ms
    {'Make': 'HONDA', 'Model': 'Accord', 'ModelYear': '2003', 'EngineCylinders': '6', 'DisplacementL': '2.998832712', 'FuelTypePrimary': 'Gasoline', 'ErrorCode': '0'}
NHTSA recalls: Corolla 2009        status 200 | 1695 ms
    recalls: 13 | first component: AIR BAGS:FRONTAL:PASSENGER SIDE:INFLATOR MODULE
NHTSA recalls: Suzuki Alto 2019    status 400 | 1136 ms
NHTSA complaints: Corolla 2009     status 200 | 5116 ms
    complaints: 1533
Open-Meteo current weather         status 200 | 729 ms
    {'time': '2026-11-21T17:15', 'interval': 900, 'temperature_2m': 13.5, 'relative_humidity_2m': 68}
fueleconomy.gov models 2009        status 200 | 1518 ms
    first models: ['4Runner 2WD', '4Runner 4WD', 'Avalon', 'Camry', 'Camry Hybrid']
fueleconomy.gov fuel prices        status 200 | 1577 ms
    {'cng': '3.1', 'diesel': '6.13', 'e85': '2.95', 'electric': '0.15', 'lpg': '3.55'}


In [42]:
import requests

def lookup_vehicle(vin):
    r = requests.get(f"https://vpic.nhtsa.dot.gov/api/vehicles/DecodeVinValues/{vin}",
                     params={"format": "json"}, timeout=25)
    if not r.ok:
        return {"ok": False, "reason": f"HTTP {r.status_code}"}
    d = r.json()["Results"][0]
    disp = d.get("DisplacementL")
    out = {"ok": d.get("ErrorCode") in ("0", "0,0"),
           "make": d.get("Make"), "model": d.get("Model"), "year": d.get("ModelYear"),
           "cylinders": int(d["EngineCylinders"]) if d.get("EngineCylinders") else None,
           "disp_L": round(float(disp), 1) if disp else None,
           "fuel": d.get("FuelTypePrimary"), "error": d.get("ErrorText")}
    return out

def recalls(make, model, year):
    r = requests.get("https://api.nhtsa.gov/recalls/recallsByVehicle",
                     params={"make": make, "model": model, "modelYear": year}, timeout=25)
    if not r.ok:
        return {"ok": False, "reason": f"no data for {make} {model} {year} (HTTP {r.status_code})"}
    j = r.json()
    return {"ok": True, "count": j.get("Count"),
            "components": sorted({x.get("Component") for x in j.get("results", [])})[:5]}

print(lookup_vehicle("1HGCM82633A004352"))
print(lookup_vehicle("NOTAVIN123"))
print(recalls("toyota", "corolla", 2009))
print(recalls("suzuki", "alto", 2019))

{'ok': True, 'make': 'HONDA', 'model': 'Accord', 'year': '2003', 'cylinders': 6, 'disp_L': 3.0, 'fuel': 'Gasoline', 'error': '0 - VIN decoded clean. Check Digit (9th position) is correct'}
{'ok': False, 'make': '', 'model': '', 'year': '', 'cylinders': None, 'disp_L': None, 'fuel': '', 'error': '6 - Incomplete VIN; 7 - Manufacturer is not registered with NHTSA for sale or importation in the U.S. for use on U.S roads; Please contact the manufacturer directly for more information; 400 - Invalid Characters Present'}
{'ok': True, 'count': 13, 'components': ['AIR BAGS:FRONTAL:DRIVER SIDE:INFLATOR MODULE', 'AIR BAGS:FRONTAL:PASSENGER SIDE:INFLATOR MODULE', 'EQUIPMENT:OTHER:LABELS', 'SEATS:FRONT ASSEMBLY:SEAT HEATER/COOLER', 'SERVICE BRAKES, HYDRAULIC:POWER ASSIST:VACUUM']}
{'ok': False, 'reason': 'no data for suzuki alto 2019 (HTTP 400)'}


In [43]:
import time
E = pd.read_csv(root / "database.csv", usecols=["Year", "Make", "Model"], low_memory=False)
E = E[E["Year"].between(2005, 2017)].drop_duplicates(["Year", "Make", "Model"]).sample(30, random_state=0)

out, t0 = [], time.time()
for _, r in E.iterrows():
    res = recalls(r["Make"].lower(), r["Model"].split()[0].lower(), int(r["Year"]))
    out.append({"year": r["Year"], "make": r["Make"], "model": r["Model"],
                "ok": res["ok"], "recalls": res.get("count")})
    time.sleep(0.5)
R = pd.DataFrame(out)
print(R.to_string(index=False))
print(f"{len(R)} calls in {time.time() - t0:.0f} s | success rate {R['ok'].mean():.0%}")

 year          make                 model    ok  recalls
 2011        Nissan          Frontier 4WD  True      1.0
 2012           Kia                  Soul  True      3.0
 2012        Suzuki             SX4 Sport  True      3.0
 2015  Aston Martin            V8 Vantage False      NaN
 2012    Mitsubishi                Lancer  True      1.0
 2015         Dodge               Dart GT  True      6.0
 2014      Chrysler                   300  True      8.0
 2012          Audi             R8 Spyder False      NaN
 2014        Nissan                Sentra  True      5.0
 2008 Mercedes-Benz CLK63 AMG (Cabriolet) False      NaN
 2017 Mercedes-Benz         GLA250 4matic  True      3.0
 2014 Mercedes-Benz      SLS AMG Roadster False      NaN
 2005       Hyundai               Elantra  True      4.0
 2014      Infiniti                  Q50a False      NaN
 2011           GMC           Terrain FWD  True      4.0
 2011          Audi          S5 Cabriolet  True      1.0
 2010           GMC            

In [44]:
import time, requests
failed = R[~R["ok"]]
for _, r in failed.iterrows():
    make, year = r["make"].lower(), int(r["year"])
    first = r["model"].split()[0].lower()
    resp = requests.get("https://api.nhtsa.gov/recalls/recallsByVehicle",
                        params={"make": make, "model": first, "modelYear": year}, timeout=25)
    try:
        m = requests.get("https://api.nhtsa.gov/products/vehicle/models",
                         params={"modelYear": year, "make": make, "issueType": "r"}, timeout=25)
        names = [x.get("model") for x in m.json().get("results", [])] if m.ok else []
    except Exception as e:
        names = [f"model list failed: {type(e).__name__}"]
    cand = [n for n in names if first[:3] in str(n).lower()]
    print(f"{year} {make} {r['model']:26s} -> '{first}' HTTP {resp.status_code} | {resp.text[:90]!r}")
    print("      NHTSA names:", cand[:4] or names[:4])
    time.sleep(0.5)
    

2015 aston martin V8 Vantage                 -> 'v8' HTTP 400 | '{"Count":0,"Message":"Results returned successfully","results":[]}'
      NHTSA names: ['V8 VANTAGE']
2012 audi R8 Spyder                  -> 'r8' HTTP 400 | '{"Count":0,"Message":"Results returned successfully","results":[]}'
      NHTSA names: ['A3', 'A5', 'A5', 'A6']
2008 mercedes-benz CLK63 AMG (Cabriolet)      -> 'clk63' HTTP 400 | '{"Count":0,"Message":"Results returned successfully","results":[]}'
      NHTSA names: ['CLK-CLASS', 'CLK-CLASS']
2014 mercedes-benz SLS AMG Roadster           -> 'sls' HTTP 400 | '{"Count":0,"Message":"Results returned successfully","results":[]}'
      NHTSA names: ['CLA-CLASS', 'CLA-CLASS', 'C-CLASS', 'C-CLASS']
2014 infiniti Q50a                       -> 'q50a' HTTP 400 | '{"Count":0,"Message":"Results returned successfully","results":[]}'
      NHTSA names: ['Q50', 'Q50', 'Q50 HYBRID', 'Q50 HYBRID']
2011 porsche Boxster S                  -> 'boxster' HTTP 400 | '{"Count":0,"Message"

In [33]:
from scipy.stats import wilcoxon

names = ["ridge", "random_forest", "hist_gb"]
print("Wilcoxon signed-rank on per-car error (paired by car)")
pairs = [("baseline_mean", n) for n in names] + [("ridge", "hist_gb"), ("random_forest", "hist_gb"), ("ridge", "random_forest")]
for a, b in pairs:
    d = (car_err[a] - car_err[b]).dropna()
    stat, p = wilcoxon(d)
    print(f"{a:14s} vs {b:14s} median diff {d.median():6.2f} pts | p = {p:.4g}")

rng = np.random.default_rng(0)
print("\nBootstrap 95% CI of the median per-car error (resampling cars)")
for n in ["baseline_mean"] + names:
    v = car_err[n].values
    meds = [np.median(rng.choice(v, len(v))) for _ in range(2000)]
    print(f"{n:14s} {np.median(v):5.2f}%  [{np.percentile(meds, 2.5):5.2f}, {np.percentile(meds, 97.5):5.2f}]")

print("\nInference time on 100,000 rows")
samp = S.sample(100_000, random_state=0)
for n in ["ridge", "random_forest", "hist_gb"]:
    m = models[n]().fit(S[FEATS], S["y"])
    t0 = time.time(); m.predict(samp[FEATS]); print(f"{n:14s} {time.time() - t0:5.2f} s")

Wilcoxon signed-rank on per-car error (paired by car)
baseline_mean  vs ridge          median diff   9.56 pts | p = 2.01e-15
baseline_mean  vs random_forest  median diff  10.69 pts | p = 4.693e-16
baseline_mean  vs hist_gb        median diff  10.06 pts | p = 3.695e-15
ridge          vs hist_gb        median diff   0.43 pts | p = 0.09355
random_forest  vs hist_gb        median diff  -0.18 pts | p = 0.2834
ridge          vs random_forest  median diff   0.74 pts | p = 0.05888

Bootstrap 95% CI of the median per-car error (resampling cars)
baseline_mean  19.19%  [14.96, 24.05]
ridge           8.45%  [ 6.66,  9.59]
random_forest   7.30%  [ 5.60,  8.69]
hist_gb         7.40%  [ 5.68,  8.73]

Inference time on 100,000 rows
ridge           0.08 s
random_forest   0.32 s
hist_gb         0.28 s


In [5]:
fr = [c for c in v.columns if "Fuel Rate" in c]
print("fuel rate columns:", fr)
if fr:
    r = v[["VehId", "MAF[g/sec]", fr[0]]].dropna()
    print("rows with both MAF and fuel rate:", len(r))
    r["formula_Lhr"] = r["MAF[g/sec]"] * 3600 / 14.7 / 750
    print("corr:", round(r["formula_Lhr"].corr(r[fr[0]]), 3))
    ratio = (r[fr[0]] / r["formula_Lhr"]).replace([np.inf, -np.inf], np.nan).dropna()
    print("measured / formula ratio:", ratio.describe()[["mean", "50%", "std"]].round(2).to_dict())

fuel rate columns: ['Fuel Rate[L/hr]']
rows with both MAF and fuel rate: 0
corr: nan
measured / formula ratio: {'mean': nan, '50%': nan, 'std': nan}


In [6]:
fr = "Fuel Rate[L/hr]"
g = v.groupby("VehId")[fr].agg(rows="count")
g = g[g["rows"] > 0].sort_values("rows", ascending=False)
print("vehicles with fuel rate:", len(g), "of", v["VehId"].nunique())
print(g.head(10))

vehicles with fuel rate: 7 of 246
       rows
VehId      
569    6925
536    3501
550    2956
537    1975
457    1793
542    1162
492     121


In [7]:
fv = v.groupby("VehId")["Fuel Rate[L/hr]"].count()
fv = fv[fv > 0].index
x = v[v["VehId"].isin(fv)]
cols = ["MAF[g/sec]", "Fuel Rate[L/hr]", "Engine RPM[RPM]", "Vehicle Speed[km/h]"]
print(x.groupby("VehId")[cols].count())
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
print(st[st["VehId"].isin(fv)].to_string())

       MAF[g/sec]  Fuel Rate[L/hr]  Engine RPM[RPM]  Vehicle Speed[km/h]
VehId                                                                   
457             0             1793             1793                 1793
492             0              121              121                  121
536             0             3501             3501                 3501
537             0             1975             1975                 1975
542             0             1162             1162                 1162
550             0             2956             2956                 2956
569             0             6925             6925                 6925
Empty DataFrame
Columns: [VehId, Vehicle Type, Vehicle Class, Engine Configuration & Displacement, Transmission, Drive Wheels, Generalized_Weight]
Index: []


In [8]:
use = ["timeStamp", "tripID", "gps_speed", "cTemp", "dtc", "eLoad",
       "kpl", "maf", "rpm", "speed", "tPos", "deviceID"]
A = pd.read_csv(root / "allcars.csv", usecols=use, low_memory=False)
print(A.shape, "devices:", A["deviceID"].nunique())
s = A.groupby("deviceID").agg(
    rows=("maf", "size"),
    maf0=("maf", lambda z: (z == 0).mean()),
    kpl0=("kpl", lambda z: (z == 0).mean()),
    maf_mean=("maf", "mean"),
)
print(s.round(2).head(20))
print(A["dtc"].value_counts().head(8))

m = A[(A["maf"] > 0) & (A["speed"] > 5)].copy()
m["kpl_formula"] = m["speed"] / (m["maf"] * 3600 / 14.7 / 750)
print("corr kpl vs formula:", round(m["kpl"].corr(m["kpl_formula"]), 3))
print((m["kpl"] / m["kpl_formula"]).describe()[["mean", "50%", "std"]].round(2).to_dict())

(7214695, 12) devices: 29
            rows  maf0  kpl0  maf_mean
deviceID                              
1         117275  1.00  1.00      0.41
2         106659  1.00  1.00      0.03
4         969210  0.01  0.03      6.20
5          85248  0.00  0.00       NaN
8          48963  0.00  0.00      0.00
11         42345  0.20  0.30     12.91
12        174486  0.12  0.23      7.64
13        160836  0.14  0.23     13.00
15        645835  0.00  0.00       NaN
16        234403  0.00  0.00    255.00
19         57767  0.00  0.00       NaN
20         13245  0.00  0.00       NaN
22        160069  0.00  0.00    255.00
23        547680  0.09  0.13     16.98
24         40139  0.00  0.00    255.00
28        158868  0.00  0.00       NaN
34         13508  0.00  0.00       NaN
35         72575  1.00  1.00      0.00
36         11885  0.00  0.00       NaN
37        323610  0.71  0.71      0.00
dtc
0.0      1093304
255.0    1005896
Name: count, dtype: int64
corr kpl vs formula: 1.0
{'mean': 0.99, '50%': 0.99,

In [9]:
m = A[A["maf"].between(0.5, 250)]
g = m.groupby("deviceID")
out = pd.DataFrame({"valid_rows": g.size(),
                    "maf_med": g["maf"].median(),
                    "speed_med": g["speed"].median()})
out["L100"] = 100 * (g["maf"].sum() / 14.7 / 750) / (g["speed"].sum() / 3600)
print(out.round(1).to_string())

          valid_rows  maf_med  speed_med     L100
deviceID                                         
1                 78    134.0      157.0     27.5
2                 20    134.0      157.0     27.9
4             834918      6.0       36.0      5.9
11             32674     14.4       26.0     19.9
12            152379      5.4       33.0      7.0
13            137240     11.9       34.0     13.0
23            424123     13.1       66.0      7.9
38              2367     48.6        0.0  35751.8
43            205236      3.8       26.0      6.8
45            438435     10.5       29.0     11.9
46            176981     23.1       55.0     14.0
47             71729      9.4       21.0     15.2
51                 9     35.4      173.0     10.0
53            617598     11.4       20.0     17.8


In [11]:
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
ice = st.loc[st["Vehicle Type"] == "ICE", "VehId"]
w = v[v["VehId"].isin(ice)]
c = w.groupby("VehId")["MAF[g/sec]"].apply(lambda z: z.notna().mean())
print("ICE vehicles in this week:", len(c), "| MAF >=50% filled:", (c >= 0.5).sum())

ICE vehicles in this week: 161 | MAF >=50% filled: 122


In [12]:
# --- VED (ICE only, MAF present) ---
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
ice = st.loc[st["Vehicle Type"] == "ICE", "VehId"]
V = v[v["VehId"].isin(ice)].copy()
V = V.sort_values(["VehId", "Trip", "Timestamp(ms)"])
V["dt"] = V.groupby(["VehId", "Trip"])["Timestamp(ms)"].diff() / 1000
V = V.rename(columns={"Vehicle Speed[km/h]": "speed", "MAF[g/sec]": "maf",
                      "Engine RPM[RPM]": "rpm", "Absolute Load[%]": "load"})
V["car"] = "ved_" + V["VehId"].astype(str)
V["source"] = "VED"
V = V[["source", "car", "dt", "speed", "maf", "rpm", "load"]]

# --- allcars (kept devices) ---
keep = [4, 12, 13, 23, 43, 45, 46, 47, 53]
C = A[A["deviceID"].isin(keep)].copy()
C["t"] = pd.to_datetime(C["timeStamp"], errors="coerce")
C = C.sort_values(["deviceID", "tripID", "t"])
C["dt"] = C.groupby(["deviceID", "tripID"])["t"].diff().dt.total_seconds()
C = C.rename(columns={"eLoad": "load"})
C["car"] = "all_" + C["deviceID"].astype(str)
C["source"] = "allcars"
C = C[["source", "car", "dt", "speed", "maf", "rpm", "load"]]

D = pd.concat([V, C], ignore_index=True)
D = D[D["dt"].between(0.1, 30) & D["maf"].between(0.5, 250) & D["speed"].notna()]
D["fuel_gps"] = D["maf"] / 14.7
D["L_step"] = D["fuel_gps"] * D["dt"] / 750 / 1000 * 1000
D["km_step"] = D["speed"] * D["dt"] / 3600
print(D.shape, "cars:", D["car"].nunique())
print(D.groupby("source").agg(rows=("dt", "size"), cars=("car", "nunique"),
                              dt_med=("dt", "median"), maf_med=("maf", "median")))

(3323130, 10) cars: 131
            rows  cars  dt_med  maf_med
source                                 
VED       267758   122     0.8     6.79
allcars  3055372     9     1.0     9.00


In [13]:
D = D.sort_values(["car"]).reset_index(drop=True)
g = D.groupby("car")

D["rpm_x_load"] = D["rpm"] * D["load"]
D["speed_sq"] = D["speed"] ** 2
D["rpm_per_kmh"] = D["rpm"] / D["speed"].clip(lower=1)
D["load_per_rpm"] = D["load"] / D["rpm"].clip(lower=1)
D["is_idle"] = (D["speed"] < 2).astype(int)
D["is_highway"] = (D["speed"] > 80).astype(int)
D["accel"] = g["speed"].diff() / D["dt"]
D["accel_abs"] = D["accel"].abs()
D["rpm_change"] = g["rpm"].diff().abs()
D["speed_roll5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=1).mean())
D["rpm_roll5"] = g["rpm"].transform(lambda z: z.rolling(5, min_periods=1).mean())
D["speed_std5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=2).std())

D = D.replace([np.inf, -np.inf], np.nan)
print(D[["rpm_x_load", "rpm_per_kmh", "accel", "speed_roll5", "speed_std5"]].describe().round(2))
print("load missing:", D["load"].isna().mean().round(3))

N = 20000
S = D.groupby("car", group_keys=False).apply(lambda z: z.iloc[::max(1, len(z)//N)].head(N))
print(S.shape, S.groupby("source")["car"].nunique().to_dict())

       rpm_x_load  rpm_per_kmh       accel  speed_roll5  speed_std5
count  3268153.00   3323130.00  3322999.00   3323130.00  3322999.00
mean     52307.92       127.91       -0.04        36.11        2.98
std      42373.40       296.47       15.84        22.64        6.84
min          0.00         0.00    -1130.00         0.00        0.00
25%      29614.31        30.98        0.00        19.60        0.00
50%      34823.53        33.33        0.00        36.00        0.71
75%      75160.78        64.32        0.00        53.80        3.13
max    1479151.57     14849.75     1270.00       157.00      124.27
load missing: 0.017


KeyError: 'Column not found: car'

In [14]:
N = 20000
D["_r"] = D.groupby("car").cumcount()
D["_n"] = D.groupby("car")["dt"].transform("size")
step = (D["_n"] // N).clip(lower=1)
S = D[D["_r"] % step == 0].copy()
S = S[S.groupby("car").cumcount() < N].drop(columns=["_r", "_n"])
print(S.shape, S.groupby("source")["car"].nunique().to_dict())
print(S.groupby("source")["car"].size())
print(S.groupby("source")["load"].median())

(447758, 22) {'VED': 122, 'allcars': 9}
source
VED        267758
allcars    180000
Name: car, dtype: int64
source
VED        23.529411
allcars    32.156863
Name: load, dtype: float64


In [ ]:
import time
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

S = S.copy()
S["accel"] = S["accel"].clip(-10, 10)
S["accel_abs"] = S["accel"].abs()
S["rpm_per_kmh"] = S["rpm_per_kmh"].clip(upper=500)
S["y"] = S["maf"] / 14.7 * 3600 / 750          # fuel, L/h
S = S.dropna(subset=["rpm", "speed", "y"]).reset_index(drop=True)
S["L_true"] = S["y"] * S["dt"] / 3600
S["km"] = S["speed"] * S["dt"] / 3600

F_NOLOAD = ["rpm", "speed", "speed_sq", "rpm_per_kmh", "is_idle", "is_highway",
            "accel", "accel_abs", "rpm_change", "speed_roll5", "rpm_roll5", "speed_std5"]
F_LOAD = F_NOLOAD + ["load", "rpm_x_load", "load_per_rpm"]

imp = lambda m: make_pipeline(SimpleImputer(strategy="median"), m)
models = {
    "baseline_mean": lambda: DummyRegressor(),
    "ridge": lambda: imp(make_pipeline(StandardScaler(), Ridge())),
    "random_forest": lambda: imp(RandomForestRegressor(
        n_estimators=60, min_samples_leaf=20, max_samples=0.3, n_jobs=-1, random_state=0)),
    "hist_gb": lambda: HistGradientBoostingRegressor(random_state=0),
}

In [15]:
import time
from sklearn.model_selection import GroupKFold
from sklearn.dummy import DummyRegressor
from sklearn.linear_model import Ridge
from sklearn.ensemble import RandomForestRegressor, HistGradientBoostingRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

S = S.copy()
S["accel"] = S["accel"].clip(-10, 10)
S["accel_abs"] = S["accel"].abs()
S["rpm_per_kmh"] = S["rpm_per_kmh"].clip(upper=500)
S["y"] = S["maf"] / 14.7 * 3600 / 750          # fuel, L/h
S = S.dropna(subset=["rpm", "speed", "y"]).reset_index(drop=True)
S["L_true"] = S["y"] * S["dt"] / 3600
S["km"] = S["speed"] * S["dt"] / 3600

F_NOLOAD = ["rpm", "speed", "speed_sq", "rpm_per_kmh", "is_idle", "is_highway",
            "accel", "accel_abs", "rpm_change", "speed_roll5", "rpm_roll5", "speed_std5"]
F_LOAD = F_NOLOAD + ["load", "rpm_x_load", "load_per_rpm"]

imp = lambda m: make_pipeline(SimpleImputer(strategy="median"), m)
models = {
    "baseline_mean": lambda: DummyRegressor(),
    "ridge": lambda: imp(make_pipeline(StandardScaler(), Ridge())),
    "random_forest": lambda: imp(RandomForestRegressor(
        n_estimators=60, min_samples_leaf=20, max_samples=0.3, n_jobs=-1, random_state=0)),
    "hist_gb": lambda: HistGradientBoostingRegressor(random_state=0),
}

In [16]:
def run(feats, label):
    gkf = GroupKFold(n_splits=5)
    out = []
    for name, mk in models.items():
        pred = np.zeros(len(S)); t0 = time.time()
        for tr, te in gkf.split(S, groups=S["car"]):
            m = mk(); m.fit(S.iloc[tr][feats], S.iloc[tr]["y"])
            pred[te] = m.predict(S.iloc[te][feats])
        secs = time.time() - t0
        T = S[["car", "L_true", "km"]].copy()
        T["L_pred"] = pred * S["dt"] / 3600
        c = T.groupby("car").sum()
        c = c[c["km"] > 0.5]
        err = ((c["L_pred"] - c["L_true"]).abs() / c["L_true"]).mean()
        out.append({"features": label, "model": name,
                    "MAE_Lh": np.abs(pred - S["y"]).mean(),
                    "car_L100_error_%": 100 * err, "secs": secs})
    return pd.DataFrame(out)

res = pd.concat([run(F_NOLOAD, "no_load"), run(F_LOAD, "with_load")])
print(res.round(2).to_string(index=False))

 features         model  MAE_Lh  car_L100_error_%   secs
  no_load baseline_mean    2.52             32.58   3.29
  no_load         ridge    1.81             27.81   8.35
  no_load random_forest    1.55             25.73  89.85
  no_load       hist_gb    1.54             26.06  19.06
with_load baseline_mean    2.52             32.58   1.68
with_load         ridge    1.78             25.94   8.80
with_load random_forest    1.39             23.18 118.10
with_load       hist_gb    1.39             23.11  20.61


In [ ]:
import re
def parse(s):
    s = str(s)
    cyl = re.match(r"(\d+)", s)
    disp = re.search(r"(\d+\.\d+)L", s)
    return (int(cyl.group(1)) if cyl else np.nan,
            float(disp.group(1)) if disp else np.nan,
            int("T/C" in s or "SC" in s))

st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
st[["cyl", "disp_L", "turbo"]] = st["Engine Configuration & Displacement"].apply(
    lambda s: pd.Series(parse(s)))
st["car"] = "ved_" + st["VehId"].astype(str)

W = S[S["source"] == "VED"].merge(st[["car", "cyl", "disp_L", "turbo"]], on="car", how="left")
print("VED cars with engine spec:", W.groupby("car")["disp_L"].first().notna().sum(),
      "of", W["car"].nunique())

base = F_LOAD
eng = F_LOAD + ["cyl", "disp_L", "turbo"]
for label, feats in [("VED no_engine", base), ("VED + engine", eng)]:
    pred = np.zeros(len(W))
    for tr, te in GroupKFold(5).split(W, groups=W["car"]):
        m = HistGradientBoostingRegressor(random_state=0).fit(W.iloc[tr][feats], W.iloc[tr]["y"])
        pred[te] = m.predict(W.iloc[te][feats])
    T = W[["car", "L_true", "km"]].copy()
    T["L_pred"] = pred * W["dt"] / 3600
    c = T.groupby("car").sum(); c = c[c["km"] > 0.5]
    err = 100 * ((c["L_pred"] - c["L_true"]).abs() / c["L_true"]).mean()
    print(f"{label}: MAE {np.abs(pred - W['y']).mean():.2f} L/h | per-car L/100km error {err:.1f}%")

In [17]:
import re
def parse(s):
    s = str(s)
    cyl = re.match(r"(\d+)", s)
    disp = re.search(r"(\d+\.\d+)L", s)
    return (int(cyl.group(1)) if cyl else np.nan,
            float(disp.group(1)) if disp else np.nan,
            int("T/C" in s or "SC" in s))

st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
st[["cyl", "disp_L", "turbo"]] = st["Engine Configuration & Displacement"].apply(
    lambda s: pd.Series(parse(s)))
st["car"] = "ved_" + st["VehId"].astype(str)

W = S[S["source"] == "VED"].merge(st[["car", "cyl", "disp_L", "turbo"]], on="car", how="left")
print("VED cars with engine spec:", W.groupby("car")["disp_L"].first().notna().sum(),
      "of", W["car"].nunique())

base = F_LOAD
eng = F_LOAD + ["cyl", "disp_L", "turbo"]
for label, feats in [("VED no_engine", base), ("VED + engine", eng)]:
    pred = np.zeros(len(W))
    for tr, te in GroupKFold(5).split(W, groups=W["car"]):
        m = HistGradientBoostingRegressor(random_state=0).fit(W.iloc[tr][feats], W.iloc[tr]["y"])
        pred[te] = m.predict(W.iloc[te][feats])
    T = W[["car", "L_true", "km"]].copy()
    T["L_pred"] = pred * W["dt"] / 3600
    c = T.groupby("car").sum(); c = c[c["km"] > 0.5]
    err = 100 * ((c["L_pred"] - c["L_true"]).abs() / c["L_true"]).mean()
    print(f"{label}: MAE {np.abs(pred - W['y']).mean():.2f} L/h | per-car L/100km error {err:.1f}%")

VED cars with engine spec: 122 of 122
VED no_engine: MAE 1.42 L/h | per-car L/100km error 23.4%
VED + engine: MAE 1.04 L/h | per-car L/100km error 8.1%


In [18]:
print(((c["L_pred"] - c["L_true"]).abs() / c["L_true"] * 100).describe().round(1))

count    121.0
mean       8.1
std        9.3
min        0.1
25%        2.7
50%        5.5
75%       10.7
max       74.9
dtype: float64


In [19]:
c["err_%"] = (c["L_pred"] - c["L_true"]).abs() / c["L_true"] * 100
bad = c.sort_values("err_%", ascending=False).head(10)
bad = bad.join(W.groupby("car")[["cyl", "disp_L", "turbo"]].first())
bad["rows"] = W.groupby("car").size()
bad["L100_true"] = 100 * bad["L_true"] / bad["km"]
print(bad.round(1))
print("share of cars under 10%:", round((c["err_%"] < 10).mean(), 2),
      "| under 15%:", round((c["err_%"] < 15).mean(), 2))

         L_true    km  L_pred  err_%   cyl  disp_L  turbo  rows  L100_true
car                                                                       
ved_282     1.5  15.5     2.6   74.9   4.0     2.5    0.0  1846        9.5
ved_466     0.3   3.5     0.4   43.2   4.0     2.0    0.0   406        8.7
ved_319     0.5   8.1     0.7   35.1   4.0     1.5    0.0   959        6.5
ved_196     3.3  28.7     4.3   28.5   6.0     4.3    0.0  3020       11.6
ved_271     0.1   0.8     0.1   21.9   4.0     2.0    0.0   144       14.7
ved_160     0.3   1.4     0.2   20.9   4.0     2.4    0.0   293       18.8
ved_116     7.8  24.6     6.2   19.7  10.0     6.8    0.0  3640       31.6
ved_340     3.3  39.0     3.9   19.1   6.0     3.5    0.0  5049        8.4
ved_213     4.2  34.0     3.4   18.6   6.0     4.2    0.0  3591       12.4
ved_313     1.0   8.5     1.2   17.5   4.0     2.4    0.0  2353       12.3
share of cars under 10%: 0.69 | under 15%: 0.87


In [20]:
print(st["turbo"].value_counts())

turbo
0.0    336
1.0     21
Name: count, dtype: int64


In [21]:
import glob
cols = ["VehId", "Trip", "Timestamp(ms)", "Vehicle Speed[km/h]",
        "MAF[g/sec]", "Engine RPM[RPM]", "Absolute Load[%]"]
ren = {"Vehicle Speed[km/h]": "speed", "MAF[g/sec]": "maf",
       "Engine RPM[RPM]": "rpm", "Absolute Load[%]": "load"}
ice = set(st.loc[st["Vehicle Type"] == "ICE", "VehId"])
files = sorted(glob.glob(str(root / "VED_DynamicData_Part*" / "*.csv")))
print(len(files), "files")

def load_week(f):
    w = pd.read_csv(f, usecols=cols).rename(columns=ren)
    w = w[w["VehId"].isin(ice)].sort_values(["VehId", "Trip", "Timestamp(ms)"])
    k = ["VehId", "Trip"]
    w["dt"] = w.groupby(k)["Timestamp(ms)"].diff() / 1000
    w = w[w["dt"].between(0.1, 30) & w["maf"].between(0.5, 250)
          & w["speed"].notna() & w["rpm"].notna()].copy()
    g = w.groupby(k)
    w["rpm_x_load"] = w["rpm"] * w["load"]
    w["speed_sq"] = w["speed"] ** 2
    w["rpm_per_kmh"] = (w["rpm"] / w["speed"].clip(lower=1)).clip(upper=500)
    w["load_per_rpm"] = w["load"] / w["rpm"].clip(lower=1)
    w["is_idle"] = (w["speed"] < 2).astype(int)
    w["is_highway"] = (w["speed"] > 80).astype(int)
    w["accel"] = (g["speed"].diff() / w["dt"]).clip(-10, 10)
    w["accel_abs"] = w["accel"].abs()
    w["rpm_change"] = g["rpm"].diff().abs()
    w["speed_roll5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=1).mean())
    w["rpm_roll5"] = g["rpm"].transform(lambda z: z.rolling(5, min_periods=1).mean())
    w["speed_std5"] = g["speed"].transform(lambda z: z.rolling(5, min_periods=2).std())
    w = w.drop(columns=["Timestamp(ms)"])
    return w.replace([np.inf, -np.inf], np.nan).astype("float32")

VA = pd.concat([load_week(f) for f in files], ignore_index=True)
VA["VehId"] = VA["VehId"].astype(int)
VA["car"] = "ved_" + VA["VehId"].astype(str)
VA["source"] = "VED"
print(VA.shape, "cars:", VA["car"].nunique())

N = 20000
VA["_r"] = VA.groupby("car").cumcount()
VA["_n"] = VA.groupby("car")["dt"].transform("size")
VA = VA[VA["_r"] % (VA["_n"] // N).clip(lower=1) == 0]
VA = VA[VA.groupby("car").cumcount() < N].drop(columns=["_r", "_n"])
print(VA.shape, VA.groupby("car").size().describe().round(0))

54 files
(10386041, 21) cars: 197
(2982294, 21) count      197.0
mean     15139.0
std       7051.0
min         71.0
25%      10131.0
50%      20000.0
75%      20000.0
max      20000.0
dtype: float64


In [1]:
import pandas as pd, numpy as np
from pathlib import Path
root = Path("Data")

E = pd.read_csv(root / "engine_data.csv")
print(E.shape)
print(E.dtypes)
print(E.head())
print("missing:", E.isna().sum().sum())

(19535, 7)
Engine rpm            int64
Lub oil pressure    float64
Fuel pressure       float64
Coolant pressure    float64
lub oil temp        float64
Coolant temp        float64
Engine Condition      int64
dtype: object
   Engine rpm  Lub oil pressure  Fuel pressure  Coolant pressure  \
0         700          2.493592      11.790927          3.178981   
1         876          2.941606      16.193866          2.464504   
2         520          2.961746       6.553147          1.064347   
3         473          3.707835      19.510172          3.727455   
4         619          5.672919      15.738871          2.052251   

   lub oil temp  Coolant temp  Engine Condition  
0     84.144163     81.632187                 1  
1     77.640934     82.445724                 0  
2     77.752266     79.645777                 1  
3     74.129907     71.774629                 1  
4     78.396989     87.000225                 0  
missing: 0


In [2]:
tcol = "Engine Condition"
print(E[tcol].value_counts(normalize=True).round(3))
print("duplicate rows:", E.duplicated().sum())
print(E.describe().T.round(2))
print(E.groupby(tcol).mean().T.round(2))

Engine Condition
1    0.631
0    0.369
Name: proportion, dtype: float64
duplicate rows: 0
                    count    mean     std    min     25%     50%     75%  \
Engine rpm        19535.0  791.24  267.61  61.00  593.00  746.00  934.00   
Lub oil pressure  19535.0    3.30    1.02   0.00    2.52    3.16    4.06   
Fuel pressure     19535.0    6.66    2.76   0.00    4.92    6.20    7.74   
Coolant pressure  19535.0    2.34    1.04   0.00    1.60    2.17    2.85   
lub oil temp      19535.0   77.64    3.11  71.32   75.73   76.82   78.07   
Coolant temp      19535.0   78.43    6.21  61.67   73.90   78.35   82.92   
Engine Condition  19535.0    0.63    0.48   0.00    0.00    1.00    1.00   

                      max  
Engine rpm        2239.00  
Lub oil pressure     7.27  
Fuel pressure       21.14  
Coolant pressure     7.48  
lub oil temp        89.58  
Coolant temp       195.53  
Engine Condition     1.00  
Engine Condition       0       1
Engine rpm        885.00  736.30
Lub oil pre

In [3]:
from sklearn.model_selection import cross_val_score
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X = E.drop(columns=[tcol]); y = E[tcol]
print("majority:", round(y.value_counts(normalize=True).max(), 3))
for name, m in [("tree depth 3", DecisionTreeClassifier(max_depth=3, random_state=0)),
                ("logistic", make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000))),
                ("random forest", RandomForestClassifier(200, n_jobs=-1, random_state=0))]:
    s = cross_val_score(m, X, y, cv=5, scoring="f1")
    print(name, "F1:", s.mean().round(3), "+/-", s.std().round(3))

rf = RandomForestClassifier(200, n_jobs=-1, random_state=0).fit(X, y)
print(pd.Series(rf.feature_importances_, X.columns).sort_values(ascending=False).round(3))

majority: 0.631
tree depth 3 F1: 0.769 +/- 0.004
logistic F1: 0.766 +/- 0.003
random forest F1: 0.747 +/- 0.006
Engine rpm          0.221
Fuel pressure       0.166
lub oil temp        0.159
Lub oil pressure    0.154
Coolant pressure    0.150
Coolant temp        0.149
dtype: float64


In [10]:
use_dev = [4, 11, 12, 13, 23, 43, 45, 46, 47, 53]
B = A[A["deviceID"].isin(use_dev)].copy()
B["t"] = pd.to_datetime(B["timeStamp"], errors="coerce")
B = B.sort_values(["deviceID", "tripID", "t"])
B["dt"] = B.groupby(["deviceID", "tripID"])["t"].diff().dt.total_seconds()
B = B[B["dt"].between(0.1, 30) & B["maf"].between(0.5, 250)]
B["L"] = B["maf"] / 14.7 / 750 * B["dt"]
B["km"] = B["speed"] * B["dt"] / 3600
s = B.groupby("deviceID").agg(rows=("dt", "size"), km=("km", "sum"), L=("L", "sum"),
                              dt_med=("dt", "median"))
s["L_per_100km"] = 100 * s["L"] / s["km"]
print(s.round(2))

            rows       km       L  dt_med  L_per_100km
deviceID                                              
4         834768  8138.24  477.01     1.0         5.86
11         32625   249.40   49.72     1.0        19.94
12        152200  1652.69  115.57     1.0         6.99
13        137036  1465.43  190.87     1.0        13.02
23        423808  6276.96  493.37     1.0         7.86
43        204842  1563.45  106.32     1.0         6.80
45        437516  4181.20  498.44     1.0        11.92
46        176911  2613.67  365.63     1.0        13.99
47         71606   514.49   78.03     1.0        15.17
53        616685  4280.94  761.99     1.0        17.80


In [ ]:
import pandas as pd, numpy as np, glob, os

folder = "Data/VED_DynamicData_Part1"
files = sorted(glob.glob(os.path.join(folder, "VED_*_week.csv")))

keep = ("DayNum","VehId","Trip","Timestamp","Vehicle Speed","MAF","Engine RPM",
        "Absolute Load","OAT","Outside","Fuel Rate","Short Term","Long Term")

def short(c):
    if c.startswith("Vehicle Speed"): return "speed"
    if c.startswith("MAF"): return "maf"
    if c.startswith("Engine RPM"): return "rpm"
    if c.startswith("Absolute Load"): return "load"
    if c.startswith(("OAT","Outside")): return "oat"
    if c.startswith("Fuel Rate"): return "fuel"
    if c.startswith("Short Term"): return "stft1" if "1" in c else "stft2"
    if c.startswith("Long Term"): return "ltft1" if "1" in c else "ltft2"
    if c.startswith("Timestamp"): return "ts_ms"
    return c

dyn = pd.concat([pd.read_csv(f, usecols=lambda c: c.startswith(keep)) for f in files],
                ignore_index=True)
dyn.columns = [short(c) for c in dyn.columns]
print(dyn.shape, dyn.columns.tolist())

static = pd.read_excel("Data/VED_Static_Data_ICE&HEV.xlsx")
print(static.columns.tolist())
et = [c for c in static.columns if "EngineType" in c.replace(" ", "")][0]
print(static[et].value_counts())

dyn = dyn.merge(static[["VehId", et]].rename(columns={et: "EngineType"}),
                on="VehId", how="left")

pos = dyn[dyn["fuel"] > 0]
print("\nPositive fuel rows:", len(pos), "| vehicles:", pos["VehId"].nunique())
print(pos.groupby("EngineType")["VehId"].nunique())
print(pos["VehId"].value_counts().head(10))

chk = pos.dropna(subset=["maf"]).copy()
chk["est"] = chk["maf"] / 14.7 / 740 * 3600      # estimated L/hr
print("\nEstimate vs measured fuel rate:")
print(chk.groupby("EngineType").apply(lambda g: pd.Series({
    "rows": len(g),
    "corr": g["fuel"].corr(g["est"]),
    "median_est_over_measured": (g["est"] / g["fuel"]).median()})))

In [ ]:
static = pd.read_excel("Data/VED_Static_Data_ICE&HEV.xlsx")
static.columns = [str(c).strip() for c in static.columns]
print(static.columns.tolist())
print(static.head(3))

vid = [c for c in static.columns if c.lower().replace(" ", "") == "vehid"][0]
et_list = ([c for c in static.columns if "engine" in c.lower() and "type" in c.lower()]
           or [c for c in static.columns if "type" in c.lower()])
et = et_list[0] if et_list else None
print("VehId column:", vid, "| EngineType column:", et)

if et:
    st = static[[vid, et]].rename(columns={vid: "VehId", et: "EngineType"})
    print(st["EngineType"].value_counts())
else:
    st = static[[vid]].rename(columns={vid: "VehId"})
    st["EngineType"] = "unknown"

dyn = dyn.merge(st, on="VehId", how="left")

pos = dyn[dyn["fuel"] > 0]
print("\nPositive fuel rows:", len(pos), "| vehicles:", pos["VehId"].nunique())
print(pos.groupby("EngineType")["VehId"].nunique())
print(pos["VehId"].value_counts().head(10))

chk = pos.dropna(subset=["maf"]).copy()
chk["est"] = chk["maf"] / 14.7 / 740 * 3600      # estimated L/hr
print("\nEstimate vs measured fuel rate:")
print(chk.groupby("EngineType").apply(lambda g: pd.Series({
    "rows": len(g),
    "corr": g["fuel"].corr(g["est"]),
    "median_est_over_measured": (g["est"] / g["fuel"]).median()})))

In [ ]:
ice_ids = set(st.loc[st["EngineType"] == "ICE", "VehId"])
ice = dyn[dyn["VehId"].isin(ice_ids) & (dyn["rpm"] > 0)]
print("ICE rows (engine running):", len(ice), "| vehicles:", ice["VehId"].nunique())

m = ice.dropna(subset=["maf"])
print("ICE rows with MAF:", len(m), "| vehicles with MAF:", m["VehId"].nunique())
print("ICE rows with MAF + load + speed:", len(ice.dropna(subset=["maf","load","speed"])))
print("Top vehicles by usable rows:\n", m["VehId"].value_counts().head(5))

v = pos.loc[pos["EngineType"] == "ICE", "VehId"].unique()
print("\nICE vehicle(s) with measured fuel:", v)
one = dyn[dyn["VehId"].isin(v)]
print(one[["maf", "fuel", "rpm"]].notna().mean().round(2))

c = one.dropna(subset=["maf", "fuel"])
c = c[c["fuel"] > 0]
if len(c):
    est = c["maf"] / 14.7 / 740 * 3600
    print("Rows with both:", len(c), "| corr:", round(c["fuel"].corr(est), 2),
          "| median est/measured:", round((est / c["fuel"]).median(), 2))
else:
    print("No rows with both MAF and measured fuel")

In [ ]:
both = dyn[(dyn["fuel"] > 0) & dyn["maf"].notna() & (dyn["rpm"] > 0)].copy()
print("rows with both:", len(both), "| vehicles:", both["VehId"].nunique())
if len(both):
    both["est"] = both["maf"] / 14.7 / 740 * 3600      # estimated L/hr
    print("overall corr:", round(both["fuel"].corr(both["est"]), 3))
    print("median est/measured:", round((both["est"] / both["fuel"]).median(), 2))
    print(both.groupby("VehId").apply(lambda g: pd.Series({
        "rows": len(g),
        "corr": g["fuel"].corr(g["est"]),
        "ratio": (g["est"] / g["fuel"]).median()})).round(2))

In [ ]:

import pandas as pd
import glob
import os

folder = "Data/VED_DynamicData_Part1"

# Find all weekly CSV files
files = glob.glob(
    os.path.join(folder, "**", "*.csv"),
    recursive=True
)

print("Total weekly files found:", len(files))

# Combine all files
df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

print("Total rows:", len(df))
print("Unique vehicles:", df["VehId"].nunique())
print("Total columns:", len(df.columns))


In [ ]:
d = ice.dropna(subset=["maf", "speed", "ts_ms"]).sort_values(["VehId", "Trip", "ts_ms"]).copy()
g = d.groupby(["VehId", "Trip"])
d["dt"] = g["ts_ms"].diff() / 1000
d["dspeed"] = g["speed"].diff()
d = d[(d["dt"] > 0) & (d["dt"] <= 10)]                      # drop gaps
d["accel"] = d["dspeed"] / d["dt"] / 3.6                    # m/s^2

# fuel estimate following the VED paper (Algorithm 1), AFR = 14.08
stft = d[["stft1", "stft2"]].mean(axis=1).fillna(0)
ltft = d[["ltft1", "ltft2"]].mean(axis=1).fillna(0)
corr = (1 + stft / 100 + ltft / 100) / 14.08
d["fuel_gps"] = d["maf"] * corr                             # g/s of fuel
d["fuel_l"] = d["fuel_gps"] * d["dt"] / 740                 # litres (assumed 740 g/L)
d["dist_km"] = d["speed"] * d["dt"] / 3600

trip = d.groupby(["VehId", "Trip"]).agg(
    duration_s=("dt", "sum"), fuel_l=("fuel_l", "sum"), dist_km=("dist_km", "sum"),
    mean_speed=("speed", "mean"), max_speed=("speed", "max"), speed_std=("speed", "std"),
    accel_std=("accel", "std"), hard_brakes=("accel", lambda a: (a < -2.5).sum()),
    hard_accels=("accel", lambda a: (a > 2.5).sum()),
    idle_share=("speed", lambda s: (s < 2).mean()),
    oat_mean=("oat", "mean"), n=("dt", "size")).reset_index()

trip = trip[(trip["dist_km"] >= 2) & (trip["duration_s"] >= 120)]
trip["l_per_100km"] = trip["fuel_l"] / trip["dist_km"] * 100
trip = trip[trip["l_per_100km"].between(2, 30)]

s = static.rename(columns={"Engine Configuration & Displacement": "engine",
                           "Generalized_Weight": "weight"})
s["disp_l"] = s["engine"].astype(str).str.extract(r"(\d+\.\d)\s*L")[0].astype(float)
s["weight"] = pd.to_numeric(s["weight"], errors="coerce")
trip = trip.merge(s[["VehId", "Vehicle Class", "disp_l", "weight"]], on="VehId", how="left")

print("MAF/load/rpm correlation:\n", d[["maf", "load", "rpm"]].corr().round(2))
print("\nTrips:", len(trip), "| vehicles:", trip["VehId"].nunique())
print(trip["l_per_100km"].describe().round(2))
print(trip.isna().mean().round(2))

In [ ]:
WIN = 180   # window length in seconds

d = d.sort_values(["VehId", "Trip", "ts_ms"]).copy()
d["t"] = d.groupby(["VehId", "Trip"])["dt"].cumsum()
d["seg"] = (d["t"] // WIN).astype(int)
d["hb"] = (d["accel"] < -2.5).astype(int)
d["ha"] = (d["accel"] > 2.5).astype(int)
d["idle"] = (d["speed"] < 2).astype(int)

seg = d.groupby(["VehId", "Trip", "seg"]).agg(
    duration_s=("dt", "sum"), fuel_l=("fuel_l", "sum"), dist_km=("dist_km", "sum"),
    mean_speed=("speed", "mean"), max_speed=("speed", "max"), speed_std=("speed", "std"),
    accel_std=("accel", "std"), hard_brakes=("hb", "sum"), hard_accels=("ha", "sum"),
    idle_share=("idle", "mean"), oat=("oat", "mean"), daynum=("DayNum", "min"),
    lat=("lat", "mean") if "lat" in d.columns else ("speed", "size")
).reset_index() if False else d.groupby(["VehId", "Trip", "seg"]).agg(
    duration_s=("dt", "sum"), fuel_l=("fuel_l", "sum"), dist_km=("dist_km", "sum"),
    mean_speed=("speed", "mean"), max_speed=("speed", "max"), speed_std=("speed", "std"),
    accel_std=("accel", "std"), hard_brakes=("hb", "sum"), hard_accels=("ha", "sum"),
    idle_share=("idle", "mean"), oat=("oat", "mean"), daynum=("DayNum", "min")
).reset_index()

seg = seg[(seg["duration_s"] >= 0.8 * WIN) & (seg["dist_km"] >= 0.5)]
seg["l_per_100km"] = seg["fuel_l"] / seg["dist_km"] * 100
seg = seg[seg["l_per_100km"].between(2, 30)]

seg["date"] = pd.Timestamp("2017-11-01") + pd.to_timedelta(seg["daynum"] - 1, unit="D")
seg["hour"] = ((seg["daynum"] % 1) * 24).astype(int)
seg["weekday"] = seg["date"].dt.dayofweek

seg = seg.merge(s[["VehId", "Vehicle Class", "disp_l", "weight"]], on="VehId", how="left")
seg["weight"] = seg["weight"].fillna(seg.groupby("Vehicle Class")["weight"].transform("median"))

print("Segments:", len(seg), "| vehicles:", seg["VehId"].nunique(), "| trips:", seg.groupby(["VehId","Trip"]).ngroups)
print(seg["l_per_100km"].describe().round(2))
print(seg.isna().mean().round(2))

In [ ]:
import pandas as pd
p = r"D:\allcars.csv"
head = pd.read_csv(p, nrows=200_000)
print(head.columns.tolist())
print(head.head(3).T)
print(head.isna().mean().round(2))

n = sum(len(c) for c in pd.read_csv(p, usecols=[0], chunksize=1_000_000))
print("total rows:", n)

In [ ]:
import os
folder = r"D:\allcars.csv"
print(os.path.isdir(folder), os.listdir(folder))
for f in os.listdir(folder):
    fp = os.path.join(folder, f)
    print(f, os.path.isfile(fp), round(os.path.getsize(fp) / 1e9, 2), "GB")

In [ ]:
import pandas as pd

p = r"D:\allcars.csv\allcars.csv"
head = pd.read_csv(p, nrows=200_000)
print(head.columns.tolist())
print(head.head(3).T)
print(head.isna().mean().round(2))

n = sum(len(c) for c in pd.read_csv(p, usecols=[0], chunksize=1_000_000))
print("total rows:", n)

In [16]:
import pandas as pd, numpy as np

p = r"D:\Ml project\Data\allcars.csv"
cols = ["timeStamp","tripID","gps_speed","battery","cTemp","dtc","eLoad","iat","imap","kpl","maf","rpm","speed","tAdv","tPos","deviceID"]
f32 = ["gps_speed","battery","cTemp","eLoad","iat","imap","kpl","maf","rpm","tAdv","tPos"]
dt = {c: "float32" for c in f32}
dt["dtc"] = "string"

lv = pd.read_csv(p, usecols=cols, dtype=dt, low_memory=False)
lv["timeStamp"] = pd.to_datetime(lv["timeStamp"], errors="coerce")

print(lv.shape)
print("time range:", lv["timeStamp"].min(), "->", lv["timeStamp"].max())
print("devices:", lv["deviceID"].nunique(), "| trips:", lv.groupby(["deviceID","tripID"]).ngroups)
print(lv["deviceID"].value_counts().head(10))

top = lv["deviceID"].value_counts().head(10).index
sub = lv[lv["deviceID"].isin(top)]
miss = sub.groupby("deviceID")[["cTemp","dtc","maf","kpl","battery"]].agg(lambda s: s.isna().mean()).round(2)
print("\nMissing share by device (top 10):")
print(miss)

print("\nDTC values:")
print(lv["dtc"].value_counts(dropna=True).head(15))
print("rows with a DTC:", lv["dtc"].notna().sum())
print("devices with a DTC:", lv.loc[lv["dtc"].notna(), "deviceID"].nunique())

print("\nDescribe:")
print(lv[["speed","gps_speed","rpm","maf","kpl","cTemp","battery","eLoad","tPos"]].describe().round(2).T)
print("maf == 0 share:", round((lv["maf"] == 0).mean(), 2))
print("kpl == 0 share:", round((lv["kpl"] == 0).mean(), 2))

c = lv[(lv["speed"] > 10) & (lv["maf"] > 0) & (lv["kpl"] > 0)].copy()
c["fuel_lph"] = c["maf"] / 14.08
c["fuel_lph"] = c["fuel_lph"] * 3600
c["fuel_lph"] = c["fuel_lph"] / 740
c["kpl_est"] = c["speed"] / c["fuel_lph"]
print("kpl check rows:", len(c))
print("corr:", round(c["kpl"].corr(c["kpl_est"]), 3))
print("median est/kpl:", round((c["kpl_est"] / c["kpl"]).median(), 2))

PermissionError: [Errno 13] Permission denied

In [17]:
paths = [r"D:\Ml project\Data\allcars.csv", r"D:\allcars.csv\allcars.csv"]
for q in paths:
    try:
        with open(q, "rb") as fh:
            fh.read(100)
        print("OK     ", q)
    except Exception as e:
        print("FAILED ", q, "->", type(e).__name__)

OK      D:\Ml project\Data\allcars.csv
OK      D:\allcars.csv\allcars.csv


In [18]:
import pandas as pd, numpy as np

p = r"D:\Ml project\Data\allcars.csv"
cols = ["timeStamp","tripID","gps_speed","battery","cTemp","dtc","eLoad","iat","imap","kpl","maf","rpm","speed","tAdv","tPos","deviceID"]
f32 = ["gps_speed","battery","cTemp","eLoad","iat","imap","kpl","maf","rpm","tAdv","tPos"]
dt = {c: "float32" for c in f32}
dt["dtc"] = "string"

lv = pd.read_csv(p, usecols=cols, dtype=dt, low_memory=False)
lv["timeStamp"] = pd.to_datetime(lv["timeStamp"], errors="coerce")

print(lv.shape)
print("time range:", lv["timeStamp"].min(), "->", lv["timeStamp"].max())
print("devices:", lv["deviceID"].nunique(), "| trips:", lv.groupby(["deviceID","tripID"]).ngroups)
print(lv["deviceID"].value_counts().head(10))

top = lv["deviceID"].value_counts().head(10).index
sub = lv[lv["deviceID"].isin(top)]
miss = sub.groupby("deviceID")[["cTemp","dtc","maf","kpl","battery"]].agg(lambda s: s.isna().mean()).round(2)
print("\nMissing share by device (top 10):")
print(miss)

print("\nDTC values:")
print(lv["dtc"].value_counts(dropna=True).head(15))
print("rows with a DTC:", lv["dtc"].notna().sum())
print("devices with a DTC:", lv.loc[lv["dtc"].notna(), "deviceID"].nunique())

print("\nDescribe:")
print(lv[["speed","gps_speed","rpm","maf","kpl","cTemp","battery","eLoad","tPos"]].describe().round(2).T)
print("maf == 0 share:", round((lv["maf"] == 0).mean(), 2))
print("kpl == 0 share:", round((lv["kpl"] == 0).mean(), 2))

c = lv[(lv["speed"] > 10) & (lv["maf"] > 0) & (lv["kpl"] > 0)].copy()
fuel_lph = c["maf"] / 14.08 * 3600 / 740
c["kpl_est"] = c["speed"] / fuel_lph
print("kpl check rows:", len(c))
print("corr:", round(c["kpl"].corr(c["kpl_est"]), 3))
print("median est/kpl:", round((c["kpl_est"] / c["kpl"]).median(), 2))

(7214695, 16)
time range: 2017-08-30 00:00:00 -> 2017-12-31 23:31:04
devices: 29 | trips: 6553
deviceID
4     969210
46    939311
53    784215
15    645835
45    576251
23    547680
43    483059
37    323610
51    246753
16    234403
Name: count, dtype: int64

Missing share by device (top 10):
          cTemp   dtc   maf   kpl  battery
deviceID                                  
4          1.00  1.00  0.12  0.12     0.12
15         1.00  1.00  1.00  1.00     1.00
16         0.00  0.00  0.00  0.00     0.00
23         0.98  0.98  0.12  0.12     0.12
37         1.00  1.00  0.29  0.29     0.29
43         0.58  0.58  0.46  0.46     0.46
45         0.39  0.39  0.00  0.00     0.00
46         0.36  0.36  0.17  0.17     0.17
51         1.00  1.00  0.99  0.99     0.99
53         0.56  0.56  0.06  0.06     0.06

DTC values:
dtc
0.0      1093304
255.0     731354
255       274542
Name: count, dtype: Int64
rows with a DTC: 2099200
devices with a DTC: 13

Describe:
               count    mean     std

In [19]:
paths = [r"D:\Ml project\Data\allcars.csv", r"D:\allcars.csv\allcars.csv"]
for q in paths:
    try:
        with open(q, "rb") as fh:
            fh.read(100)
        print("OK     ", q)
    except Exception as e:
        print("FAILED ", q, "->", type(e).__name__)

OK      D:\Ml project\Data\allcars.csv
OK      D:\allcars.csv\allcars.csv


In [20]:
print(lv.shape)
print("devices:", lv["deviceID"].nunique())
print("trips:", lv.groupby(["deviceID","tripID"]).ngroups)
print(lv["timeStamp"].min(), lv["timeStamp"].max())
print(lv["deviceID"].value_counts().head(10))
tot = lv.groupby("deviceID").size()
has = lv.groupby("deviceID")[["cTemp","dtc","battery","maf"]].count().div(tot, axis=0).round(2)
print("devices with coolant temp >50% of rows:", (has["cTemp"] > 0.5).sum())
print("devices with battery >50% of rows:", (has["battery"] > 0.5).sum())
print("devices with MAF >50% of rows:", (has["maf"] > 0.5).sum())
print(lv["dtc"].value_counts(dropna=True).head(15))
print("rows with a DTC:", lv["dtc"].notna().sum())
print("devices with a DTC:", lv.loc[lv["dtc"].notna(), "deviceID"].nunique())

(7214695, 16)
devices: 29
trips: 6553
2017-08-30 00:00:00 2017-12-31 23:31:04
deviceID
4     969210
46    939311
53    784215
15    645835
45    576251
23    547680
43    483059
37    323610
51    246753
16    234403
Name: count, dtype: int64
devices with coolant temp >50% of rows: 7
devices with battery >50% of rows: 18
devices with MAF >50% of rows: 18
dtc
0.0      1093304
255.0     731354
255       274542
Name: count, dtype: Int64
rows with a DTC: 2099200
devices with a DTC: 13


In [22]:
import pandas as pd
import os

folder = r"D:\MI project\Data"

files = [
    "exp1_14drivers_14cars_dailyRoutes.csv",
    "exp2_19drivers_1car_1route.csv",
    "exp3_4drivers_1car_1route.csv"
]

for file in files:
    path = os.path.join(folder, file)
    df = pd.read_csv(path, low_memory=False)

    print("\n", "=" * 60)
    print("FILE:", file)
    print("SHAPE:", df.shape)
    print("ROWS:", len(df))
    print("COLUMNS:", len(df.columns))
    print("\nCOLUMN NAMES:")
    print(df.columns.tolist())
    print("\nMISSING VALUES (%):")
    print((df.isnull().mean() * 100).round(2))
    print("\nFIRST 3 ROWS:")
    print(df.head(3))

PermissionError: [Errno 13] Permission denied: 'D:\\MI project\\Data\\exp1_14drivers_14cars_dailyRoutes.csv'

In [23]:
import os

folder = r"D:\MI project\Data"

print("Folder exists:", os.path.exists(folder))
print("Is directory:", os.path.isdir(folder))
print("\nContents:")

if os.path.isdir(folder):
    print(os.listdir(folder))

Folder exists: True
Is directory: True

Contents:
['exp1_14drivers_14cars_dailyRoutes.csv']


In [24]:
import pandas as pd
import os

file_path = r"D:\MI project\Data\exp1_14drivers_14cars_dailyRoutes.csv"

df = pd.read_csv(file_path, low_memory=False)

print("Shape:", df.shape)
print("Rows:", len(df))
print("Columns:", len(df.columns))
print("\nColumn names:")
print(df.columns.tolist())
print("\nFirst 5 rows:")
display(df.head())

PermissionError: [Errno 13] Permission denied: 'D:\\MI project\\Data\\exp1_14drivers_14cars_dailyRoutes.csv'

In [25]:
import os

file_path = r"D:\MI project\Data\exp1_14drivers_14cars_dailyRoutes.csv"

print("File exists:", os.path.isfile(file_path))
print("File size:", os.path.getsize(file_path), "bytes")

try:
    with open(file_path, "rb") as f:
        print("Python can access the file!")
        print("First 100 bytes:", f.read(100))
except Exception as e:
    print(type(e).__name__, ":", e)

File exists: False
File size: 0 bytes
PermissionError : [Errno 13] Permission denied: 'D:\\MI project\\Data\\exp1_14drivers_14cars_dailyRoutes.csv'


In [ ]:
import os
import stat

folder = r"D:\MI project\Data"
filename = "exp1_14drivers_14cars_dailyRoutes.csv"
path = os.path.join(folder, filename)

print("Absolute path:", os.path.abspath(path))
print("Exists:", os.path.exists(path))
print("Is file:", os.path.isfile(path))
print("Is directory:", os.path.isdir(path))
print("Is link:", os.path.islink(path))

try:
    print("File information:", os.stat(path))
except Exception as e:
    print("Stat error:", repr(e))

In [27]:
import os
import stat

folder = r"D:\MI project\Data"
filename = "exp1_14drivers_14cars_dailyRoutes.csv"
path = os.path.join(folder, filename)

print("Absolute path:", os.path.abspath(path))
print("Exists:", os.path.exists(path))
print("Is file:", os.path.isfile(path))
print("Is directory:", os.path.isdir(path))
print("Is link:", os.path.islink(path))

try:
    print("File information:", os.stat(path))
except Exception as e:
    print("Stat error:", repr(e))

Absolute path: D:\MI project\Data\exp1_14drivers_14cars_dailyRoutes.csv
Exists: True
Is file: False
Is directory: True
Is link: False
File information: os.stat_result(st_mode=16895, st_ino=5629499534292317, st_dev=7527366604151314400, st_nlink=1, st_uid=0, st_gid=0, st_size=0, st_atime=1794842634, st_mtime=1794842634, st_ctime=1794842634)


In [ ]:
import os

path = r"D:\MI project\Data\exp1_14drivers_14cars_dailyRoutes.csv"

print("Contents inside:")
for item in os.listdir(path):
    print(item)

In [28]:

# 1. Check missing values across the complete dataset

missing = (df.isnull().mean() * 100).round(2)

print("Missing percentage per column:")
print(missing.sort_values(ascending=False))


# 2. Check fuel-rate availability

fuel_col = "Fuel Rate[L/hr]"

print("\nTotal records:", len(df))
print("Records with fuel data:", df[fuel_col].notna().sum())
print("Missing fuel records:", df[fuel_col].isna().sum())

print(
    "Fuel data availability:",
    round(df[fuel_col].notna().mean() * 100, 2), "%"
)


# 3. Identify vehicles with fuel readings

fuel_by_vehicle = (
    df.groupby("VehId")[fuel_col]
    .agg(["count", "mean", "min", "max"])
    .sort_values("count", ascending=False)
)

fuel_by_vehicle = fuel_by_vehicle[
    fuel_by_vehicle["count"] > 0
]

print("\nVehicles with fuel readings:", len(fuel_by_vehicle))
print(fuel_by_vehicle.head(20))


# 4. Check zero fuel readings

print(
    "\nZero fuel readings:",
    (df[fuel_col] == 0).sum()
)


NameError: name 'df' is not defined

In [29]:
import os

path = r"D:\MI project\Data\exp1_14drivers_14cars_dailyRoutes.csv"

print("Files and folders inside:")

for item in os.listdir(path):
    full_path = os.path.join(path, item)

    if os.path.isfile(full_path):
        print("FILE:", item)
    elif os.path.isdir(full_path):
        print("FOLDER:", item)

Files and folders inside:
FILE: exp1_14drivers_14cars_dailyRoutes.csv


In [ ]:
import os

folder = r"D:\MI project\Data"

for root, dirs, files in os.walk(folder):
    print("\nLOCATION:", root)

    for name in dirs:
        print("FOLDER:", repr(name))

    for name in files:
        print("FILE:", repr(name))

In [30]:

# Investigate zero and positive fuel readings

fuel = df["Fuel Rate[L/hr]"]

print("Fuel rate distribution:")
print(fuel.describe())

# Compare vehicle speed and RPM
print("\nZERO fuel readings:")
print(
    df.loc[fuel == 0, [
        "Vehicle Speed[km/h]",
        "Engine RPM[RPM]"
    ]].describe()
)

print("\nPOSITIVE fuel readings:")
print(
    df.loc[fuel > 0, [
        "Vehicle Speed[km/h]",
        "Engine RPM[RPM]"
    ]].describe()
)

# Check how many positive fuel readings each vehicle has
positive_fuel = (
    df[df["Fuel Rate[L/hr]"] > 0]
    .groupby("VehId")
    .size()
    .sort_values(ascending=False)
)

print("\nPositive fuel readings per vehicle:")
print(positive_fuel)


NameError: name 'df' is not defined

In [31]:
import os

folder = r"D:\MI project\Data"

for root, dirs, files in os.walk(folder):
    print("\nCurrent location:", root)

    for name in dirs:
        print("FOLDER:", name)

    for name in files:
        print("FILE:", name)


Current location: D:\MI project\Data
FOLDER: exp1_14drivers_14cars_dailyRoutes.csv

Current location: D:\MI project\Data\exp1_14drivers_14cars_dailyRoutes.csv
FILE: exp1_14drivers_14cars_dailyRoutes.csv


In [32]:
import pandas as pd

path = r"D:\MI project\Data\exp1_14drivers_14cars_dailyRoutes.csv\exp1_14drivers_14cars_dailyRoutes.csv"
df = pd.read_csv(path)
print(df.shape)
df.head()

D:\temp\ipykernel_12804\2431907830.py:4: DtypeWarning: Columns (1: MARK, 2: MODEL, 4: ENGINE_POWER, 5: AUTOMATIC, 6: VEHICLE_ID, 9: FUEL_LEVEL, 10: ENGINE_LOAD, 14: MAF, 15: LONG TERM FUEL TRIM BANK 2, 16: FUEL_TYPE, 20: SHORT TERM FUEL TRIM BANK 2, 21: SHORT TERM FUEL TRIM BANK 1, 22: ENGINE_RUNTIME, 23: THROTTLE_POS, 24: DTC_NUMBER, 25: TROUBLE_CODES, 26: TIMING_ADVANCE, 27: EQUIV_RATIO) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(path)


(60439, 33)


,TIMESTAMP,MARK,MODEL,CAR_YEAR,ENGINE_POWER,AUTOMATIC,VEHICLE_ID,BAROMETRIC_PRESSURE(KPA),ENGINE_COOLANT_TEMP,FUEL_LEVEL,...,THROTTLE_POS,DTC_NUMBER,TROUBLE_CODES,TIMING_ADVANCE,EQUIV_RATIO,MIN,HOURS,DAYS_OF_WEEK,MONTHS,YEAR
0,1.502903e+12,chevrolet,agile,2011.0,"1,4",n,car1,100.0,80.0,"48,60%",...,25%,MIL is OFF0 codes,NaN,"56,9%","1,0%",13.0,16.0,2.0,8.0,2017.0
1,1.502903e+12,chevrolet,agile,2011.0,"1,4",n,car1,100.0,80.0,"48,60%",...,25%,MIL is OFF0 codes,NaN,"56,5%","1,0%",13.0,16.0,2.0,8.0,2017.0
2,1.502903e+12,chevrolet,agile,2011.0,"1,4",n,car1,100.0,80.0,"48,60%",...,25%,MIL is OFF0 codes,NaN,"57,3%","1,0%",13.0,16.0,2.0,8.0,2017.0
3,1.502903e+12,chevrolet,agile,2011.0,"1,4",n,car1,100.0,80.0,"48,60%",...,25%,MIL is OFF0 codes,NaN,"56,5%","1,0%",13.0,16.0,2.0,8.0,2017.0
4,1.502903e+12,chevrolet,agile,2011.0,"1,4",n,car1,100.0,80.0,"48,60%",...,25%,MIL is OFF0 codes,NaN,"56,9%","1,0%",13.0,16.0,2.0,8.0,2017.0


In [33]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)

root = Path(r"D:\MI project\Data")   # use the path from Copy Path if different

rows = []
for p in root.rglob("*"):
    if p.is_file():
        rows.append({"file": str(p.relative_to(root)), "MB": round(p.stat().st_size / 1e6, 2)})

inv = pd.DataFrame(rows).sort_values("file").reset_index(drop=True)
print(inv.to_string())

                                                                          file    MB
0  exp1_14drivers_14cars_dailyRoutes.csv\exp1_14drivers_14cars_dailyRoutes.csv  7.05


In [34]:
from pathlib import Path
import os

print("Working dir:", os.getcwd())

root = Path.cwd() / "Data"
print("Data folder:", root, "| exists:", root.exists())

for p in sorted(root.rglob("*")):
    if p.is_file():
        print(f"{p.relative_to(root)}  ({p.stat().st_size/1e6:.2f} MB)")

Working dir: d:\Ml project
Data folder: d:\Ml project\Data | exists: True
allcars.csv  (1682.62 MB)
exp1_14drivers_14cars_dailyRoutes.csv  (7.05 MB)
exp2_19drivers_1car_1route.csv  (1.52 MB)
exp3_4drivers_1car_1route.csv  (0.28 MB)
VED_DynamicData_Part1\VED_171101_week.csv  (67.31 MB)
VED_DynamicData_Part1\VED_171108_week.csv  (75.92 MB)
VED_DynamicData_Part1\VED_171115_week.csv  (86.24 MB)
VED_DynamicData_Part1\VED_171122_week.csv  (68.14 MB)
VED_DynamicData_Part1\VED_171129_week.csv  (75.97 MB)
VED_DynamicData_Part1\VED_171206_week.csv  (84.50 MB)
VED_DynamicData_Part1\VED_171213_week.csv  (96.68 MB)
VED_DynamicData_Part1\VED_171220_week.csv  (77.29 MB)
VED_DynamicData_Part1\VED_171227_week.csv  (45.14 MB)
VED_DynamicData_Part1\VED_180103_week.csv  (56.99 MB)
VED_DynamicData_Part1\VED_180110_week.csv  (78.97 MB)
VED_DynamicData_Part1\VED_180117_week.csv  (75.88 MB)
VED_DynamicData_Part1\VED_180124_week.csv  (65.07 MB)
VED_DynamicData_Part1\VED_180131_week.csv  (68.79 MB)
VED_DynamicD

In [35]:
import pandas as pd
from pathlib import Path
pd.set_option("display.max_columns", None); pd.set_option("display.width", 200)
root = Path.cwd() / "Data"

for name in ["exp1_14drivers_14cars_dailyRoutes.csv",
             "exp2_19drivers_1car_1route.csv",
             "exp3_4drivers_1car_1route.csv"]:
    d = pd.read_csv(root / name, decimal=",", low_memory=False)
    print("\n" + "=" * 70); print(name, d.shape)
    print(d.columns.tolist())
    display(d.head(3))


exp1_14drivers_14cars_dailyRoutes.csv (60439, 33)
['TIMESTAMP', 'MARK', 'MODEL', 'CAR_YEAR', 'ENGINE_POWER', 'AUTOMATIC', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE(KPA)', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE', 'MAF', 'LONG TERM FUEL TRIM BANK 2', 'FUEL_TYPE', 'AIR_INTAKE_TEMP', 'FUEL_PRESSURE', 'SPEED', 'SHORT TERM FUEL TRIM BANK 2', 'SHORT TERM FUEL TRIM BANK 1', 'ENGINE_RUNTIME', 'THROTTLE_POS', 'DTC_NUMBER', 'TROUBLE_CODES', 'TIMING_ADVANCE', 'EQUIV_RATIO', 'MIN', 'HOURS', 'DAYS_OF_WEEK', 'MONTHS', 'YEAR']


,TIMESTAMP,MARK,MODEL,CAR_YEAR,ENGINE_POWER,AUTOMATIC,VEHICLE_ID,BAROMETRIC_PRESSURE(KPA),ENGINE_COOLANT_TEMP,FUEL_LEVEL,ENGINE_LOAD,AMBIENT_AIR_TEMP,ENGINE_RPM,INTAKE_MANIFOLD_PRESSURE,MAF,LONG TERM FUEL TRIM BANK 2,FUEL_TYPE,AIR_INTAKE_TEMP,FUEL_PRESSURE,SPEED,SHORT TERM FUEL TRIM BANK 2,SHORT TERM FUEL TRIM BANK 1,ENGINE_RUNTIME,THROTTLE_POS,DTC_NUMBER,TROUBLE_CODES,TIMING_ADVANCE,EQUIV_RATIO,MIN,HOURS,DAYS_OF_WEEK,MONTHS,YEAR
0,1.502903e+12,chevrolet,agile,2011.0,1.4,n,car1,100.0,80.0,"48,60%","33,30%",NaN,1009.0,49.0,4.49,NaN,Biodiesel_Ethanol,59.0,NaN,0.0,NaN,NaN,00:03:28,25%,MIL is OFF0 codes,NaN,"56,9%","1,0%",13.0,16.0,2.0,8.0,2017.0
1,1.502903e+12,chevrolet,agile,2011.0,1.4,n,car1,100.0,80.0,"48,60%","32,50%",NaN,1003.0,52.0,4.51,NaN,Biodiesel_Ethanol,59.0,NaN,0.0,NaN,NaN,00:03:35,25%,MIL is OFF0 codes,NaN,"56,5%","1,0%",13.0,16.0,2.0,8.0,2017.0
2,1.502903e+12,chevrolet,agile,2011.0,1.4,n,car1,100.0,80.0,"48,60%","32,90%",NaN,995.0,51.0,4.48,NaN,Biodiesel_Ethanol,59.0,NaN,0.0,NaN,NaN,00:03:43,25%,MIL is OFF0 codes,NaN,"57,3%","1,0%",13.0,16.0,2.0,8.0,2017.0



exp2_19drivers_1car_1route.csv (8261, 28)
['TIME', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE', 'MAF', 'Term Fuel Trim Bank 1', 'FUEL_ECONOMY', 'Long Term Fuel Trim Bank 2', 'FUEL_TYPE', 'AIR_INTAKE_TEMP', 'FUEL_PRESSURE', 'SPEED', 'Short Term Fuel Trim Bank 2', 'Short Term Fuel Trim Bank 1', 'ENGINE_RUNTIME', 'THROTTLE_POS', 'DTC_NUMBER', 'TROUBLE_CODES', 'TIMING_ADVANCE', 'EQUIV_RATIO']


,TIME,LATITUDE,LONGITUDE,ALTITUDE,VEHICLE_ID,BAROMETRIC_PRESSURE,ENGINE_COOLANT_TEMP,FUEL_LEVEL,ENGINE_LOAD,AMBIENT_AIR_TEMP,ENGINE_RPM,INTAKE_MANIFOLD_PRESSURE,MAF,Term Fuel Trim Bank 1,FUEL_ECONOMY,Long Term Fuel Trim Bank 2,FUEL_TYPE,AIR_INTAKE_TEMP,FUEL_PRESSURE,SPEED,Short Term Fuel Trim Bank 2,Short Term Fuel Trim Bank 1,ENGINE_RUNTIME,THROTTLE_POS,DTC_NUMBER,TROUBLE_CODES,TIMING_ADVANCE,EQUIV_RATIO
0,1.513360e+12,-58.321.307,-3.520.429.461,46.0,s11,100kPa,82C,"74,50%","79,20%",28C,2124RPM,61kPa,"24,77g/s",NaN,NaN,NaN,NaN,36C,NaN,48km/h,NaN,"-1,60%",00:03:02,"34,90%",MIL is OFF0 codes,NaN,"66,70%","1,00%"
1,1.513360e+12,-58.323.103,-3.520.380.397,52.0,s11,100kPa,83C,"74,10%","78,40%",28C,2617RPM,44kPa,"30,96g/s",NaN,NaN,NaN,NaN,36C,NaN,60km/h,NaN,"-4,70%",00:03:08,"36,10%",MIL is OFF0 codes,NaN,"68,20%","1,00%"
2,1.513360e+12,-583.251.443,-3.520.323.709,49.0,s11,100kPa,83C,"73,70%","42,00%",28C,3005RPM,35kPa,"18,58g/s",NaN,NaN,NaN,NaN,36C,NaN,64km/h,NaN,"-10,20%",00:03:12,"32,20%",MIL is OFF0 codes,NaN,"75,70%","1,00%"



exp3_4drivers_1car_1route.csv (1743, 30)
['ord', 'TIME', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE', 'MAF', 'TERM FUEL TRIM BANK 1', 'FUEL_ECONOMY', 'LONG TERM FUEL TRIM BANK 2', 'FUEL_TYPE', 'AIR_INTAKE_TEMP', 'FUEL_PRESSURE', 'SPEED', 'SHORT TERM FUEL TRIM BANK 2', 'SHORT TERM FUEL TRIM BANK 1', 'ENGINE_RUNTIME', 'THROTTLE_POS', 'DTC_NUMBER', 'TROUBLE_CODES', 'TIMING_ADVANCE', 'EQUIV_RATIO', 'aqui']


,ord,TIME,LATITUDE,LONGITUDE,ALTITUDE,VEHICLE_ID,BAROMETRIC_PRESSURE,ENGINE_COOLANT_TEMP,FUEL_LEVEL,ENGINE_LOAD,AMBIENT_AIR_TEMP,ENGINE_RPM,INTAKE_MANIFOLD_PRESSURE,MAF,TERM FUEL TRIM BANK 1,FUEL_ECONOMY,LONG TERM FUEL TRIM BANK 2,FUEL_TYPE,AIR_INTAKE_TEMP,FUEL_PRESSURE,SPEED,SHORT TERM FUEL TRIM BANK 2,SHORT TERM FUEL TRIM BANK 1,ENGINE_RUNTIME,THROTTLE_POS,DTC_NUMBER,TROUBLE_CODES,TIMING_ADVANCE,EQUIV_RATIO,aqui
0,1,1.510000e+12,-583.206.639,-3.520.573.159,42.0,control1,100.0,95.0,56.1,43.1,4.0,816.0,39,4.73,NaN,NaN,NaN,NaN,34.0,NaN,NaN,NaN,0.8,00:45:33,21.2,MIL is OFF0 codes,NaN,52.9,1.0,NaN
1,2,1.510000e+12,-583.206.639,-3.520.573.159,42.0,control1,100.0,95.0,56.1,42.0,4.0,807.0,37,4.73,NaN,NaN,NaN,NaN,34.0,NaN,0.0,NaN,0.8,00:45:37,21.6,MIL is OFF0 codes,NaN,52.5,1.0,NaN
2,3,1.510000e+12,-583.201.429,-3.520.570.624,44.0,control1,100.0,94.0,56.1,50.6,4.0,1559.0,37,11.51,NaN,NaN,NaN,NaN,32.0,NaN,9.0,NaN,-6.2,00:45:41,29.8,MIL is OFF0 codes,NaN,56.9,1.0,NaN


In [36]:
import numpy as np

e1 = pd.read_csv(root / "exp1_14drivers_14cars_dailyRoutes.csv", decimal=",", low_memory=False)

def pct(s):
    return pd.to_numeric(s.astype(str).str.replace("%", "", regex=False)
                          .str.replace(",", ".", regex=False), errors="coerce")

for c in ["FUEL_LEVEL", "ENGINE_LOAD", "THROTTLE_POS", "TIMING_ADVANCE", "EQUIV_RATIO"]:
    e1[c] = pct(e1[c])

e1["runtime_s"] = pd.to_timedelta(e1["ENGINE_RUNTIME"], errors="coerce").dt.total_seconds()
e1["dt"] = e1.groupby("VEHICLE_ID")["runtime_s"].diff()
e1["dt"] = e1["dt"].where((e1["dt"] > 0) & (e1["dt"] < 30))

# fuel estimate (adjust AFR and density for the fuel blend)
AFR, DENSITY = 13.1, 0.75           # AFR (air kg per fuel kg), fuel density kg/L
e1["fuel_gps"] = e1["MAF"] * e1["EQUIV_RATIO"] / AFR
e1["fuel_Lph"] = e1["fuel_gps"] * 3.6 / DENSITY
e1["L_per_100km"] = np.where(e1["SPEED"] > 5, e1["fuel_Lph"] / e1["SPEED"] * 100, np.nan)

# 1) fill rate and sampling
print((e1.notna().mean() * 100).round(1).sort_values().head(12))
print(e1["dt"].describe())

# 2) plausibility
for col, (lo, hi) in {"SPEED": (0, 200), "ENGINE_RPM": (0, 7500), "ENGINE_LOAD": (0, 100),
                      "THROTTLE_POS": (0, 100), "FUEL_LEVEL": (0, 100), "MAF": (0, 300)}.items():
    s = e1[col]; print(f"{col}: min={s.min()}, max={s.max()}, outside range={((s<lo)|(s>hi)).mean():.2%}")

# 3) physics: MAF should rise with RPM x load
print("corr(MAF, RPM*LOAD):", e1["MAF"].corr(e1["ENGINE_RPM"] * e1["ENGINE_LOAD"]))

# 4) fuel result per car (expect roughly 6-15 L/100km)
print(e1.groupby("VEHICLE_ID")["L_per_100km"].median().round(1))

# 5) timestamp vs HOURS: should be a constant offset (timezone)
ts = pd.to_datetime(e1["TIMESTAMP"], unit="ms")
print((ts.dt.hour - e1["HOURS"]).value_counts().head())

FUEL_PRESSURE                0.2
FUEL_LEVEL                   5.0
AMBIENT_AIR_TEMP             6.0
L_per_100km                 16.4
BAROMETRIC_PRESSURE(KPA)    16.9
dt                          19.3
fuel_gps                    19.5
fuel_Lph                    19.5
MAF                         19.6
TROUBLE_CODES               19.7
ENGINE_RUNTIME              19.7
runtime_s                   19.7
dtype: float64
count    11641.000000
mean         9.008247
std          2.487723
min          1.000000
25%          8.000000
50%          8.000000
75%         11.000000
max         29.000000
Name: dt, dtype: float64
SPEED: min=0.0, max=143.0, outside range=0.00%
ENGINE_RPM: min=438.0, max=3816.0, outside range=0.00%
ENGINE_LOAD: min=5.1, max=100.0, outside range=0.00%
THROTTLE_POS: min=1.0, max=87.0, outside range=0.00%
FUEL_LEVEL: min=5.1, max=81.2, outside range=0.00%
MAF: min=0.26, max=57.2, outside range=0.00%
corr(MAF, RPM*LOAD): 0.9599537857920966
VEHICLE_ID
car1     8.3
car10    NaN
car11  

In [38]:
key = ["MAF", "ENGINE_RPM", "SPEED", "THROTTLE_POS", "ENGINE_LOAD", "EQUIV_RATIO", "FUEL_LEVEL"]
print(e1.groupby("VEHICLE_ID")[key].apply(lambda g: g.notna().mean().round(2)))

             MAF  ENGINE_RPM  SPEED  THROTTLE_POS  ENGINE_LOAD  EQUIV_RATIO  FUEL_LEVEL
VEHICLE_ID                                                                             
car1        0.20        0.20   1.00          0.21         0.20         0.20        0.21
car10       0.00        0.71   0.86          1.00         1.00         0.00        0.00
car11       0.98        0.98   0.99          0.98         0.98         0.98        0.00
car12       0.96        0.96   0.94          1.00         0.96         0.99        0.00
car13       0.00        0.87   0.89          0.72         0.91         0.00        0.00
car14       0.00        1.00   0.99          0.99         0.80         0.00        0.00
car2        0.00        1.00   0.93          0.29         1.00         0.00        1.00
car3        0.00        0.98   0.97          1.00         1.00         0.00        0.00
car4        0.00        0.98   0.99          1.00         0.98         0.00        0.00
car5        0.00        1.00   0

In [39]:
def clean(df):
    for c in df.columns:
        if df[c].dtype == object and df[c].astype(str).str.contains("%").any():
            df[c] = pd.to_numeric(df[c].astype(str).str.replace("%", "", regex=False)
                                   .str.replace(",", ".", regex=False), errors="coerce")
    return df

for name in ["exp2_19drivers_1car_1route.csv", "exp3_4drivers_1car_1route.csv"]:
    d = clean(pd.read_csv(root / name, decimal=",", low_memory=False))
    print("\n" + "=" * 70); print(name, d.shape)
    print(d.columns.tolist())
    display(d.head(3))
    print((d.notna().mean() * 100).round(1).sort_values().head(15))
    print(d.nunique().sort_values().head(12))   # shows which column identifies the driver/trip


exp2_19drivers_1car_1route.csv (8261, 28)
['TIME', 'LATITUDE', 'LONGITUDE', 'ALTITUDE', 'VEHICLE_ID', 'BAROMETRIC_PRESSURE', 'ENGINE_COOLANT_TEMP', 'FUEL_LEVEL', 'ENGINE_LOAD', 'AMBIENT_AIR_TEMP', 'ENGINE_RPM', 'INTAKE_MANIFOLD_PRESSURE', 'MAF', 'Term Fuel Trim Bank 1', 'FUEL_ECONOMY', 'Long Term Fuel Trim Bank 2', 'FUEL_TYPE', 'AIR_INTAKE_TEMP', 'FUEL_PRESSURE', 'SPEED', 'Short Term Fuel Trim Bank 2', 'Short Term Fuel Trim Bank 1', 'ENGINE_RUNTIME', 'THROTTLE_POS', 'DTC_NUMBER', 'TROUBLE_CODES', 'TIMING_ADVANCE', 'EQUIV_RATIO']


,TIME,LATITUDE,LONGITUDE,ALTITUDE,VEHICLE_ID,BAROMETRIC_PRESSURE,ENGINE_COOLANT_TEMP,FUEL_LEVEL,ENGINE_LOAD,AMBIENT_AIR_TEMP,ENGINE_RPM,INTAKE_MANIFOLD_PRESSURE,MAF,Term Fuel Trim Bank 1,FUEL_ECONOMY,Long Term Fuel Trim Bank 2,FUEL_TYPE,AIR_INTAKE_TEMP,FUEL_PRESSURE,SPEED,Short Term Fuel Trim Bank 2,Short Term Fuel Trim Bank 1,ENGINE_RUNTIME,THROTTLE_POS,DTC_NUMBER,TROUBLE_CODES,TIMING_ADVANCE,EQUIV_RATIO
0,1.513360e+12,-58.321.307,-3.520.429.461,46.0,s11,100kPa,82C,"74,50%","79,20%",28C,2124RPM,61kPa,"24,77g/s",NaN,NaN,NaN,NaN,36C,NaN,48km/h,NaN,"-1,60%",00:03:02,"34,90%",MIL is OFF0 codes,NaN,"66,70%","1,00%"
1,1.513360e+12,-58.323.103,-3.520.380.397,52.0,s11,100kPa,83C,"74,10%","78,40%",28C,2617RPM,44kPa,"30,96g/s",NaN,NaN,NaN,NaN,36C,NaN,60km/h,NaN,"-4,70%",00:03:08,"36,10%",MIL is OFF0 codes,NaN,"68,20%","1,00%"
2,1.513360e+12,-583.251.443,-3.520.323.709,49.0,s11,100kPa,83C,"73,70%","42,00%",28C,3005RPM,35kPa,"18,58g/s",NaN,NaN,NaN,NaN,36C,NaN,64km/h,NaN,"-10,20%",00:03:12,"32,20%",MIL is OFF0 codes,NaN,"75,70%","1,00%"


Long Term Fuel Trim Bank 2      0.0
Term Fuel Trim Bank 1           0.0
FUEL_ECONOMY                    0.0
FUEL_TYPE                       0.0
TROUBLE_CODES                   0.0
Short Term Fuel Trim Bank 2     0.0
FUEL_PRESSURE                   0.0
SPEED                          97.4
AIR_INTAKE_TEMP                98.6
BAROMETRIC_PRESSURE            98.8
Short Term Fuel Trim Bank 1    99.5
DTC_NUMBER                     99.5
ENGINE_LOAD                    99.5
FUEL_LEVEL                     99.5
TIMING_ADVANCE                 99.5
dtype: float64
Long Term Fuel Trim Bank 2      0
Term Fuel Trim Bank 1           0
FUEL_ECONOMY                    0
FUEL_TYPE                       0
TROUBLE_CODES                   0
Short Term Fuel Trim Bank 2     0
FUEL_PRESSURE                   0
EQUIV_RATIO                     3
DTC_NUMBER                      4
BAROMETRIC_PRESSURE             5
TIME                           14
VEHICLE_ID                     19
dtype: int64

exp3_4drivers_1car_1rou

,ord,TIME,LATITUDE,LONGITUDE,ALTITUDE,VEHICLE_ID,BAROMETRIC_PRESSURE,ENGINE_COOLANT_TEMP,FUEL_LEVEL,ENGINE_LOAD,AMBIENT_AIR_TEMP,ENGINE_RPM,INTAKE_MANIFOLD_PRESSURE,MAF,TERM FUEL TRIM BANK 1,FUEL_ECONOMY,LONG TERM FUEL TRIM BANK 2,FUEL_TYPE,AIR_INTAKE_TEMP,FUEL_PRESSURE,SPEED,SHORT TERM FUEL TRIM BANK 2,SHORT TERM FUEL TRIM BANK 1,ENGINE_RUNTIME,THROTTLE_POS,DTC_NUMBER,TROUBLE_CODES,TIMING_ADVANCE,EQUIV_RATIO,aqui
0,1,1.510000e+12,-583.206.639,-3.520.573.159,42.0,control1,100.0,95.0,56.1,43.1,4.0,816.0,39,4.73,NaN,NaN,NaN,NaN,34.0,NaN,NaN,NaN,0.8,00:45:33,21.2,MIL is OFF0 codes,NaN,52.9,1.0,NaN
1,2,1.510000e+12,-583.206.639,-3.520.573.159,42.0,control1,100.0,95.0,56.1,42.0,4.0,807.0,37,4.73,NaN,NaN,NaN,NaN,34.0,NaN,0.0,NaN,0.8,00:45:37,21.6,MIL is OFF0 codes,NaN,52.5,1.0,NaN
2,3,1.510000e+12,-583.201.429,-3.520.570.624,44.0,control1,100.0,94.0,56.1,50.6,4.0,1559.0,37,11.51,NaN,NaN,NaN,NaN,32.0,NaN,9.0,NaN,-6.2,00:45:41,29.8,MIL is OFF0 codes,NaN,56.9,1.0,NaN


FUEL_ECONOMY                    0.0
TERM FUEL TRIM BANK 1           0.0
FUEL_TYPE                       0.0
aqui                            0.0
SHORT TERM FUEL TRIM BANK 2     0.0
FUEL_PRESSURE                   0.0
LONG TERM FUEL TRIM BANK 2      0.0
TROUBLE_CODES                   3.6
EQUIV_RATIO                    47.4
TIMING_ADVANCE                 47.4
DTC_NUMBER                     94.0
SPEED                          95.0
AIR_INTAKE_TEMP                98.5
BAROMETRIC_PRESSURE            98.7
THROTTLE_POS                   99.4
dtype: float64
FUEL_ECONOMY                   0
TERM FUEL TRIM BANK 1          0
FUEL_TYPE                      0
aqui                           0
SHORT TERM FUEL TRIM BANK 2    0
FUEL_PRESSURE                  0
LONG TERM FUEL TRIM BANK 2     0
TIME                           1
DTC_NUMBER                     1
TROUBLE_CODES                  1
EQUIV_RATIO                    2
VEHICLE_ID                     4
dtype: int64


In [40]:
import numpy as np

def load(name):
    d = pd.read_csv(root / name, low_memory=False)
    skip = {"TIME", "LATITUDE", "LONGITUDE", "VEHICLE_ID", "FUEL_TYPE", "ENGINE_RUNTIME",
            "DTC_NUMBER", "TROUBLE_CODES", "aqui", "ord"}
    for c in d.columns:
        if c not in skip:
            d[c] = num(d[c])
    if "ord" in d.columns:
        d = d.sort_values(["VEHICLE_ID", "ord"])
    d["runtime_s"] = pd.to_timedelta(d["ENGINE_RUNTIME"], errors="coerce").dt.total_seconds()
    d["dt"] = d.groupby("VEHICLE_ID")["runtime_s"].diff()
    d["dt"] = d["dt"].where((d["dt"] > 0) & (d["dt"] < 30))
    return d

AFR, DENSITY = 14.7, 0.75      # set to match the fuel used

for name in ["exp2_19drivers_1car_1route.csv", "exp3_4drivers_1car_1route.csv"]:
    d = load(name)
    print("\n" + "=" * 70); print(name, d.shape)
    print("TIME unique per driver:", d.groupby("VEHICLE_ID")["TIME"].nunique().max())
    print("dt:", d["dt"].describe()[["count", "mean", "50%", "max"]].round(1).to_dict())
    print("corr(MAF, RPM*LOAD):", round(d["MAF"].corr(d["ENGINE_RPM"] * d["ENGINE_LOAD"]), 3))
    if "aqui" in d.columns: print("aqui:", d["aqui"].dropna().unique()[:5])

    d["fuel_gps"] = d["MAF"] * d["EQUIV_RATIO"] / AFR
    d["km_step"] = d["SPEED"] * d["dt"] / 3600
    d["L_step"] = d["fuel_gps"] * d["dt"] / 1000 / DENSITY
    s = d.groupby("VEHICLE_ID").agg(rows=("dt", "size"), min=("dt", lambda x: x.sum() / 60),
                                    km=("km_step", "sum"), L=("L_step", "sum"))
    s["L_per_100km"] = s["L"] / s["km"] * 100
    print(s.round(2))

NameError: name 'num' is not defined

In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

   # exp2/eroot = Path(r"D:\Ml project\Data")xp3 sit directly in Data

def num(s):
    # strip units/text like "12.5 %" or "3,4" and convert to float
    s = s.astype(str).str.replace(",", ".", regex=False)
    s = s.str.extract(r"(-?\d+\.?\d*)", expand=False)
    return pd.to_numeric(s, errors="coerce")

def load(name):
    d = pd.read_csv(root / name, low_memory=False)
    skip = {"TIME", "LATITUDE", "LONGITUDE", "VEHICLE_ID", "FUEL_TYPE", "ENGINE_RUNTIME",
            "DTC_NUMBER", "TROUBLE_CODES", "aqui", "ord"}
    for c in d.columns:
        if c not in skip:
            d[c] = num(d[c])
    if "ord" in d.columns:
        d = d.sort_values(["VEHICLE_ID", "ord"])
    d["runtime_s"] = pd.to_timedelta(d["ENGINE_RUNTIME"], errors="coerce").dt.total_seconds()
    d["dt"] = d.groupby("VEHICLE_ID")["runtime_s"].diff()
    d["dt"] = d["dt"].where((d["dt"] > 0) & (d["dt"] < 30))
    return d

AFR, DENSITY = 14.7, 0.75

for name in ["exp2_19drivers_1car_1route.csv", "exp3_4drivers_1car_1route.csv"]:
    d = load(name)
    print("\n" + "=" * 70); print(name, d.shape)
    print("TIME unique per driver:", d.groupby("VEHICLE_ID")["TIME"].nunique().max())
    print("dt:", d["dt"].describe()[["count", "mean", "50%", "max"]].round(1).to_dict())
    print("corr(MAF, RPM*LOAD):", round(d["MAF"].corr(d["ENGINE_RPM"] * d["ENGINE_LOAD"]), 3))
    if "aqui" in d.columns: print("aqui:", d["aqui"].dropna().unique()[:5])

    d["fuel_gps"] = d["MAF"] * d["EQUIV_RATIO"] / AFR
    d["km_step"] = d["SPEED"] * d["dt"] / 3600
    d["L_step"] = d["fuel_gps"] * d["dt"] / 1000 / DENSITY
    s = d.groupby("VEHICLE_ID").agg(rows=("dt", "size"), min=("dt", lambda x: x.sum() / 60),
                                    km=("km_step", "sum"), L=("L_step", "sum"))
    s["L_per_100km"] = s["L"] / s["km"] * 100
    print(s.round(2))

FileNotFoundError: [Errno 2] No such file or directory: 'D:\\MI project\\Data\\exp2_19drivers_1car_1route.csv'

In [42]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---- locate the Data folder (note: "Ml project" = lowercase L, not capital I) ----
for cand in [Path("Data"), Path(r"D:\Ml project\Data")]:
    if (cand / "exp2_19drivers_1car_1route.csv").exists():
        root = cand
        break
else:
    raise FileNotFoundError("exp2 file not found. Check the Data folder path.")
print("Using:", root.resolve())

AFR, DENSITY = 14.7, 0.75      # air-fuel ratio, fuel density (kg/L); set to match the fuel used

def num(s):
    # strip units/text like "12.5 %" or "3,4" and convert to float
    s = s.astype(str).str.replace(",", ".", regex=False)
    s = s.str.extract(r"(-?\d+\.?\d*)", expand=False)
    return pd.to_numeric(s, errors="coerce")

def load(name):
    d = pd.read_csv(root / name, low_memory=False)
    skip = {"TIME", "LATITUDE", "LONGITUDE", "VEHICLE_ID", "FUEL_TYPE", "ENGINE_RUNTIME",
            "DTC_NUMBER", "TROUBLE_CODES", "aqui", "ord"}
    for c in d.columns:
        if c not in skip:
            d[c] = num(d[c])
    if "ord" in d.columns:
        d = d.sort_values(["VEHICLE_ID", "ord"])
    d["runtime_s"] = pd.to_timedelta(d["ENGINE_RUNTIME"], errors="coerce").dt.total_seconds()
    d["dt"] = d.groupby("VEHICLE_ID")["runtime_s"].diff()
    d["dt"] = d["dt"].where((d["dt"] > 0) & (d["dt"] < 30))
    return d

for name in ["exp2_19drivers_1car_1route.csv", "exp3_4drivers_1car_1route.csv"]:
    d = load(name)
    print("\n" + "=" * 70); print(name, d.shape)
    print("TIME unique per driver:", d.groupby("VEHICLE_ID")["TIME"].nunique().max())
    print("dt:", d["dt"].describe()[["count", "mean", "50%", "max"]].round(1).to_dict())
    print("corr(MAF, RPM*LOAD):", round(d["MAF"].corr(d["ENGINE_RPM"] *

_IncompleteInputError: incomplete input (3208964866.py, line 41)

In [43]:
import numpy as np
import pandas as pd
from pathlib import Path

# ---- locate the Data folder (note: "Ml project" = lowercase L, not capital I) ----
for cand in [Path("Data"), Path(r"D:\Ml project\Data")]:
    if (cand / "exp2_19drivers_1car_1route.csv").exists():
        root = cand
        break
else:
    raise FileNotFoundError("exp2 file not found. Check the Data folder path.")
print("Using:", root.resolve())

AFR, DENSITY = 14.7, 0.75      # air-fuel ratio, fuel density (kg/L); set to match the fuel used

def num(s):
    # strip units/text like "12.5 %" or "3,4" and convert to float
    s = s.astype(str).str.replace(",", ".", regex=False)
    s = s.str.extract(r"(-?\d+\.?\d*)", expand=False)
    return pd.to_numeric(s, errors="coerce")

def load(name):
    d = pd.read_csv(root / name, low_memory=False)
    skip = {"TIME", "LATITUDE", "LONGITUDE", "VEHICLE_ID", "FUEL_TYPE", "ENGINE_RUNTIME",
            "DTC_NUMBER", "TROUBLE_CODES", "aqui", "ord"}
    for c in d.columns:
        if c not in skip:
            d[c] = num(d[c])
    if "ord" in d.columns:
        d = d.sort_values(["VEHICLE_ID", "ord"])
    d["runtime_s"] = pd.to_timedelta(d["ENGINE_RUNTIME

SyntaxError: unterminated string literal (detected at line 31) (389436994.py, line 31)

In [44]:
import numpy as np
import pandas as pd
from pathlib import Path

for cand in [Path("Data"), Path(r"D:\Ml project\Data")]:
    if (cand / "exp2_19drivers_1car_1route.csv").exists():
        root = cand
        break
else:
    raise FileNotFoundError("exp2 file not found")
print("Using:", root.resolve())

AFR, DENSITY = 14.7, 0.75

def num(s):
    s = s.astype(str).str.replace(",", ".", regex=False)
    s = s.str.extract(r"(-?\d+\.?\d*)", expand=False)
    return pd.to_numeric(s, errors="coerce")

Using: D:\Ml project\Data


In [45]:
SKIP = {"TIME", "LATITUDE", "LONGITUDE", "VEHICLE_ID", "FUEL_TYPE",
        "ENGINE_RUNTIME", "DTC_NUMBER", "TROUBLE_CODES", "aqui", "ord"}

def load(name):
    d = pd.read_csv(root / name, low_memory=False)
    for c in d.columns:
        if c not in SKIP:
            d[c] = num(d[c])
    if "ord" in d.columns:
        d = d.sort_values(["VEHICLE_ID", "ord"])
    rt = pd.to_timedelta(d["ENGINE_RUNTIME"], errors="coerce")
    d["runtime_s"] = rt.dt.total_seconds()
    g = d.groupby("VEHICLE_ID")["runtime_s"]
    d["dt"] = g.diff()
    d["dt"] = d["dt"].where((d["dt"] > 0) & (d["dt"] < 30))
    return d

In [46]:
files = ["exp2_19drivers_1car_1route.csv",
         "exp3_4drivers_1car_1route.csv"]

for name in files:
    d = load(name)
    print("\n" + "=" * 70)
    print(name, d.shape)

    t = d.groupby("VEHICLE_ID")["TIME"].nunique().max()
    print("TIME unique per driver:", t)

    st = d["dt"].describe()[["count", "mean", "50%", "max"]]
    print("dt:", st.round(1).to_dict())

    x = d["ENGINE_RPM"] * d["ENGINE_LOAD"]
    print("corr(MAF, RPM*LOAD):", round(d["MAF"].corr(x), 3))

    if "aqui" in d.columns:
        print("aqui:", d["aqui"].dropna().unique()[:5])

    d["fuel_gps"] = d["MAF"] * d["EQUIV_RATIO"] / AFR
    d["km_step"] = d["SPEED"] * d["dt"] / 3600
    d["L_step"] = d["fuel_gps"] * d["dt"] / 1000 / DENSITY

    s = d.groupby("VEHICLE_ID").agg(
        rows=("dt", "size"),
        min=("dt", lambda v: v.sum() / 60),
        km=("km_step", "sum"),
        L=("L_step", "sum"),
    )
    s["L_per_100km"] = s["L"] / s["km"] * 100
    print(s.round(2))


exp2_19drivers_1car_1route.csv (8261, 30)
TIME unique per driver: 2
dt: {'count': 8144.0, 'mean': 4.3, '50%': 4.0, 'max': 29.0}
corr(MAF, RPM*LOAD): 0.948
            rows    min     km     L  L_per_100km
VEHICLE_ID                                       
s1           313  31.27  18.99  1.95        10.29
s10          470  31.25  18.55  1.98        10.65
s11          536  35.63  18.49  2.05        11.11
s12          675  44.57  19.54  2.30        11.76
s13          537  37.62  15.03  2.24        14.94
s14          451  30.02  18.78  2.06        10.99
s15          625  41.37  18.15  2.27        12.49
s16          488  32.47  18.49  1.77         9.59
s17          179  17.53  10.91  0.99         9.05
s18          391  33.05  17.34  2.05        11.82
s19          435  28.68  18.44  1.97        10.70
s2           493  32.72  18.86  2.08        11.03
s3           134  15.78  10.60  1.35        12.77
s4           451  30.15  18.75  2.43        12.94
s5           461  30.77  19.00  2.05        

In [47]:
d = load("exp3_4drivers_1car_1route.csv")
g = d.groupby("VEHICLE_ID")
out = g[["MAF", "EQUIV_RATIO", "SPEED", "ENGINE_RPM"]].agg(
    lambda v: v.notna().mean()
)
out["MAF_mean"] = g["MAF"].mean()
out["EQ_mean"] = g["EQUIV_RATIO"].mean()
print(out.round(2))

             MAF  EQUIV_RATIO  SPEED  ENGINE_RPM  MAF_mean  EQ_mean
VEHICLE_ID                                                         
control1    1.00         0.65   0.88        1.00      6.81     1.00
control2    1.00         0.05   0.99        1.00     10.55     1.00
control3    0.99         0.20   0.99        1.00     16.61     0.97
control4    1.00         0.97   0.98        0.99     37.39     0.73


In [49]:
d = load("exp3_4drivers_1car_1route.csv")
d["EQUIV_RATIO"] = d["EQUIV_RATIO"].fillna(1.0)
d["fuel_gps"] = d["MAF"] * d["EQUIV_RATIO"] / AFR
d["km_step"] = d["SPEED"] * d["dt"] / 3600
d["L_step"] = d["fuel_gps"] * d["dt"] / 1000 / DENSITY
s = d.groupby("VEHICLE_ID").agg(
    km=("km_step", "sum"),
    L=("L_step", "sum"),
)
s["L_per_100km"] = s["L"] / s["km"] * 100
print(s.round(2))

cols = ["MAF", "ENGINE_RPM", "SPEED", "ENGINE_LOAD"]
print(d.groupby("VEHICLE_ID")[cols].median().round(1))

               km     L  L_per_100km
VEHICLE_ID                          
control1    15.82  1.66        10.47
control2    18.22  1.64         8.98
control3    18.89  2.16        11.41
control4    24.89  2.35         9.44
             MAF  ENGINE_RPM  SPEED  ENGINE_LOAD
VEHICLE_ID                                      
control1     5.9      1127.0   29.0         40.0
control2     9.0      1526.0   45.0         40.4
control3    10.1      1963.0   53.0         40.8
control4    17.2      3295.5   65.0         44.3


In [50]:
d = load("exp2_19drivers_1car_1route.csv")
kw = ["fault", "fail", "label", "dtc", "status", "health", "class"]
cols = [c for c in d.columns if any(k in c.lower() for k in kw)]
print(cols)
for c in cols:
    print(c, d[c].notna().mean().round(3), d[c].dropna().unique()[:10])

['DTC_NUMBER']
DTC_NUMBER 0.995 <StringArray>
['MIL is OFF0 codes', '4101000761001:00410100076100', '4101000761601:00410100076160', '1:6032000007E806']
Length: 4, dtype: str


In [51]:
for f in ["exp1_14drivers_14cars_dailyRoutes.csv",
          "exp2_19drivers_1car_1route.csv",
          "exp3_4drivers_1car_1route.csv"]:
    d = pd.read_csv(root / f, usecols=["DTC_NUMBER"], low_memory=False)
    print(f)
    print(d["DTC_NUMBER"].value_counts().head(8), "\n")

exp1_14drivers_14cars_dailyRoutes.csv
DTC_NUMBER
MIL is OFF0 codes      41064
MIL is OFF1 codes       6071
MIL is OFF107 codes        2
MIL is OFF16 codes         2
MIL is ON0 codes           1
MIL is OFF65 codes         1
MIL is OFF8 codes          1
Name: count, dtype: int64 

exp2_19drivers_1car_1route.csv
DTC_NUMBER
MIL is OFF0 codes               7787
4101000761001:00410100076100     406
4101000761601:00410100076160      20
1:6032000007E806                   3
Name: count, dtype: int64 

exp3_4drivers_1car_1route.csv
DTC_NUMBER
MIL is OFF0 codes    1638
Name: count, dtype: int64 



In [52]:
import pandas as pd, numpy as np
from pathlib import Path

root = Path("Data")
f = root / "VED_DynamicData_Part1" / "VED_171108_week.csv"
v = pd.read_csv(f, low_memory=False)

print(v.shape)
print(list(v.columns))
print("\nfill rate:")
print(v.notna().mean().round(3).to_string())
print("\nvehicles:", v["VehId"].nunique(), "trips:", v.groupby("VehId")["Trip"].nunique().sum())

(535198, 22)
['DayNum', 'VehId', 'Trip', 'Timestamp(ms)', 'Latitude[deg]', 'Longitude[deg]', 'Vehicle Speed[km/h]', 'MAF[g/sec]', 'Engine RPM[RPM]', 'Absolute Load[%]', 'OAT[DegC]', 'Fuel Rate[L/hr]', 'Air Conditioning Power[kW]', 'Air Conditioning Power[Watts]', 'Heater Power[Watts]', 'HV Battery Current[A]', 'HV Battery SOC[%]', 'HV Battery Voltage[V]', 'Short Term Fuel Trim Bank 1[%]', 'Short Term Fuel Trim Bank 2[%]', 'Long Term Fuel Trim Bank 1[%]', 'Long Term Fuel Trim Bank 2[%]']

fill rate:
DayNum                            1.000
VehId                             1.000
Trip                              1.000
Timestamp(ms)                     1.000
Latitude[deg]                     1.000
Longitude[deg]                    1.000
Vehicle Speed[km/h]               0.999
MAF[g/sec]                        0.804
Engine RPM[RPM]                   0.981
Absolute Load[%]                  0.691
OAT[DegC]                         0.153
Fuel Rate[L/hr]                   0.034
Air Conditioning

In [53]:
st = pd.read_excel(root / "VED_Static_Data_ICE&HEV.xlsx")
print(st.shape)
print(st.head(8).to_string())
for c in st.columns:
    if st[c].nunique() < 15:
        print("\n", c, st[c].value_counts(dropna=False).to_dict())

(357, 7)
   VehId Vehicle Type Vehicle Class Engine Configuration & Displacement    Transmission Drive Wheels Generalized_Weight
0      2          ICE           Car                       4-FI 2.0L T/C         NO DATA      NO DATA               3500
1      5          HEV           Car                 4-GAS/ELECTRIC 2.0L         NO DATA      NO DATA               3500
2      7          ICE           SUV                           6-FI 3.6L       AUTOMATIC      NO DATA               4500
3      8          ICE           Car                           4-FI 1.5L     5-SP MANUAL      NO DATA               2500
4     12          ICE           Car                           4-FI 1.8L         NO DATA      NO DATA               2500
5    108          ICE       NO DATA                        8-4V/FI 6.0L  6-SP AUTOMATIC      NO DATA               6000
6    110          ICE       NO DATA                           6-FI 3.6L         NO DATA      NO DATA               4500
7    115          HEV       NO 

In [1]:
a = pd.read_csv(root / "allcars.csv", low_memory=False, nrows=200000)
print(a.shape, "(first 200k rows)")
print(list(a.columns))
print(a.head(3).T.to_string())
print(a.notna().mean().round(2).to_string())

NameError: name 'pd' is not defined

In [2]:
import pandas as pd, numpy as np
from pathlib import Path

root = Path("Data")
v = pd.read_csv(root / "VED_DynamicData_Part1" / "VED_171108_week.csv", low_memory=False)
print(root.resolve(), v.shape)

D:\Ml project\Data (535198, 22)


In [ ]:

fuel = df["Fuel Rate[L/hr]"]
rpm = df["Engine RPM[RPM]"]

# Cross-tabulation of fuel rate and engine RPM
conditions = pd.crosstab(
    fuel > 0,
    rpm > 0,
    margins=True
)

print(conditions)

# Calculate percentages
print("\nPercentage breakdown:")
print(
    (pd.crosstab(
        fuel > 0,
        rpm > 0,
        normalize="all"
    ) * 100).round(2)
)


In [ ]:

import pandas as pd
import glob

files = glob.glob("your_dataset_folder/*.csv")

df = pd.concat(
    [pd.read_csv(file) for file in files],
    ignore_index=True
)

# Count fuel readings for every vehicle
fuel_stats = (
    df.groupby("VehId")["Fuel Rate[L/hr]"]
    .agg(
        total_rows="size",
        available_fuel="count"
    )
)

# Keep vehicles with fuel readings
fuel_stats = fuel_stats[
    fuel_stats["available_fuel"] > 0
].sort_values("available_fuel", ascending=False)

print(fuel_stats)
print("\nTotal vehicles:", len(fuel_stats))
print("Total fuel readings:", fuel_stats["available_fuel"].sum())
